# CinderTrace — Sri Lanka multi-year observed-activity validation

This notebook runs the frozen **54-date, 2018–2024** study in three resumable batches, followed by one results download. Public catalogue records, original metadata pages, source hashes and standard archive observations are embedded. No input upload or FIRMS key is needed. Earthdata login is required for new science downloads.

**Read the resource note:** the linked fire/geolocation files total approximately **45.6 GiB of downloads**. Processing stages one granule's new raw files at a time in temporary storage, then removes only those temporary copies after checkpointing. Original pilot caches are preserved. Derived clear-centre evidence and receipts stay in Drive for resuming; allow several GiB of additional Drive space and at least a couple of GiB of temporary working space. Actual space/time depends on the files and service. A runtime interruption can require re-downloading the current incomplete granule, but completed derived checkpoints are reused.

The two unresolved early NOAA-20 dates (2018-04-19 and 2018-05-27) are explicitly unavailable when the fire product is absent. The notebook does not invent files or substitute a different sensor/version. They remain in status tables, and are excluded from affected training comparisons.

This is regional research on observed activity, not a production data release. It does not modify the frontend, fill actual gaps, restore zero months or validate early-history/annual uncertainty.

## 1. Install, then restart
Run this cell once. Then select **Runtime → Restart session** in Colab, or restart the VS Code kernel. Continue at **step 2**, without rerunning installation. This avoids mixing already-loaded NumPy/SciPy libraries with newly installed files. Drive data is preserved.

In [7]:
%pip -q install numpy pandas pyarrow shapely h3 scipy netCDF4 earthaccess
print("Restart the session/kernel now, then continue from step 2.")


Restart the session/kernel now, then continue from step 2.


## 2. Check the numerical environment
If this fails, restart first and rerun from here. Do not patch private NumPy functions.

In [1]:
import sys,json,hashlib,base64,zlib,zipfile,types
from pathlib import Path
try:
    import numpy as np
    import numpy.testing
    import pandas as pd
    from scipy.spatial import cKDTree
    from scipy.optimize import nnls,linear_sum_assignment
    from netCDF4 import Dataset
    import h3
except (ImportError,AttributeError) as error:
    raise RuntimeError('Numerical imports failed. Restart the session/kernel after installation, then resume from step 2.') from error
print('Numerical imports passed. Python:',sys.version.split()[0])


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Numerical imports passed. Python: 3.13.18


## 3. Set persistent and temporary storage
In Colab, the defaults use your existing pilot folder and a new study folder in Drive. `SCRATCH` is temporary runtime storage for new raw files; it is separate from the preserved pilot cache. In VS Code, update `PILOT_ROOT` and `STUDY_ROOT` if needed. Do not put `SCRATCH` inside an existing source cache. The notebook never deletes files from `ORIGINAL_CACHE`.

In [2]:
import tempfile
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PILOT_ROOT=Path('/content/drive/MyDrive/CinderTrace_Observation_Evidence_v1')
    STUDY_ROOT=Path('/content/drive/MyDrive/CinderTrace_Multiyear_Validation_v1')
    SCRATCH=Path('/content/cindertrace_multiyear_temporary_raw')
else:
    PILOT_ROOT=Path('data/atlas/observation_evidence_run_v1')
    STUDY_ROOT=Path('data/atlas/multiyear_validation_run_v1')
    SCRATCH=Path(tempfile.gettempdir())/'cindertrace_multiyear_temporary_raw'
ORIGINAL_CACHE=PILOT_ROOT/'science_cache'
STUDY_ROOT.mkdir(parents=True,exist_ok=True);SCRATCH.mkdir(parents=True,exist_ok=True)
assert not SCRATCH.resolve().is_relative_to(ORIGINAL_CACHE.resolve()),'Choose a separate scratch directory.'
print('Study workspace:',STUDY_ROOT.resolve())
print('Temporary raw staging:',SCRATCH.resolve())
print('Preserved optional source cache:',ORIGINAL_CACHE.resolve())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Study workspace: /content/drive/MyDrive/CinderTrace_Multiyear_Validation_v1
Temporary raw staging: /content/cindertrace_multiyear_temporary_raw
Preserved optional source cache: /content/drive/MyDrive/CinderTrace_Observation_Evidence_v1/science_cache


## 4. Verify the frozen inputs
Fit dates are in 2018–2020, calibration in 2021–2022 and evaluation in 2023–2024. Dates were selected using calendar strata and availability, without ranking fire counts. The first quarter of 2018 has no eligible NOAA-20 archive dates and remains unsupported. Exact source pages and hashes are retained. Earlier aggregate scores and pilot evidence have already been inspected; this is not a blind external replication.

In [3]:
INPUT_BYTES=zlib.decompress(base64.b64decode('...'))
assert hashlib.sha256(INPUT_BYTES).hexdigest()=='54f41272f1a1139ee5f5bf72a8a48a5e5131025e89cde40e768c54cd23f9c4c5'
INPUTS=json.loads(INPUT_BYTES)
assert len(INPUTS['design']['dates'])==54
assert hashlib.sha256(INPUTS['boundary'].encode()).hexdigest()=='838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e'
for name,text in INPUTS['source_metadata_files'].items():
    receipt=INPUTS['inventory_summary']['files'][name]
    assert len(text.encode())==receipt['bytes'] and hashlib.sha256(text.encode()).hexdigest()==receipt['sha256'],name
(STUDY_ROOT/'verified_inputs.json').write_bytes(INPUT_BYTES)
print('Dates:',INPUTS['design']['date_counts_by_partition'])
print('Linked catalogue volume GiB:',round(INPUTS['inventory_summary']['matched_download_catalogue_gib'],2))
print('Unresolved dates:',INPUTS['inventory_summary']['early_dates_needing_source_reconciliation'])


Dates: {'calibration': 16, 'evaluation': 16, 'fit': 22}
Linked catalogue volume GiB: 45.57
Unresolved dates: [{'date': '2018-04-19', 'global_cmr_hits': 0, 'native_archive_detections': 2, 'product': 'VJ114IMG'}, {'date': '2018-05-27', 'global_cmr_hits': 0, 'native_archive_detections': 0, 'product': 'VJ114IMG'}]


## 5. Load the frozen processing and evaluation methods
Per-granule checkpoints preserve native pixels, QA, coordinate/type matching, viewing geometry, clear-centre evidence and download receipts. Native archive reconciliation must succeed before vegetation counts are available. Completed date checkpoints are hashed and checked on reruns.

**Two separate analyses:** shared-support diagnostics evaluate every temporal candidate before assigning geometry-admissible one-to-one pass pairs. They never select by fire counts. The simulated-outage predictor receives **only NOAA-20 screened vegetation active cells and calendar month**. It cannot use S-NPP masks, geometry or shared-support selection, which would be unavailable during a real outage.

The target is S-NPP-observed, quality-screened vegetation active-cell-days on the fixed regional H3 resolution-8 grid. Type labels are carried from the standard archive, not independently confirmed burning type. Zero requires archive reconciliation and some nominal-clear support; missing products, failed processing and absent clear support remain unavailable. This target differs from raw pixel counts and complete fire activity.

Model selection uses fit-year holdouts only, with a global ratio, shrunk seasonal ratio or nonnegative affine model. Calibration is separate. Exploratory 80/90% date intervals require at least twelve supported calibration dates. Sensitivity rows and pixels are not independent evaluation samples. There are no nominal coverage guarantees under temporal dependence, and no monthly or annual interval claims.

In [4]:
BASE_SOURCE='"""Small VIIRS standard-data observation evidence pilot; never fills archive zeros.\nPublic metadata discovery runs without credentials. Science screening requires\nEarthdata-authorized files and is a pixel-centre diagnostic, not area completeness.\n"""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport hashlib,json,re,time,urllib.request,urllib.parse,urllib.error\nimport numpy as np\nimport pandas as pd\n\nROOT=Path(__file__).resolve().parent.parent\nOUT=ROOT/\'data/atlas/observation_evidence_pilot_v1\'\nPRODUCTS={\n \'VNP14IMG\':{\'collection\':\'C2734202914-LPCLOUD\',\'version\':\'002\',\'satellite\':\'Suomi-NPP\',\'kind\':\'fire\'},\n \'VJ114IMG\':{\'collection\':\'C2734197957-LPCLOUD\',\'version\':\'002\',\'satellite\':\'NOAA-20\',\'kind\':\'fire\'},\n \'VNP03IMG\':{\'collection\':\'C2105092163-LAADS\',\'version\':\'2\',\'satellite\':\'Suomi-NPP\',\'kind\':\'geolocation\'},\n \'VJ103IMG\':{\'collection\':\'C2105086226-LAADS\',\'version\':\'2.1\',\'satellite\':\'NOAA-20\',\'kind\':\'geolocation\'}}\nDAYS={\'2021-08-15\':\'recorded_zero_control_not_no_fire_truth\',\n \'2021-08-19\':\'positive_control\', \'2022-08-04\':\'listed_SNPP_missing_standard_date\',\n \'2022-08-12\':\'post_interruption_recovery_candidate\',\n \'2024-05-31\':\'listed_SNPP_missing_standard_date\',\n \'2024-06-05\':\'post_interruption_recovery_candidate\'}\nBOUNDARY_SHA=\'838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e\'\nSOURCES={\n \'fire_guide\':\'https://ladsweb.modaps.eosdis.nasa.gov/archive/Document%20Archive/Science%20Data%20Product%20Documentation/VIIRS_C2_AF-375m_User_Guide_1.0.pdf\',\n \'snpp_geolocation\':\'https://ladsweb.modaps.eosdis.nasa.gov/missions-and-measurements/products/VNP03IMG/\',\n \'noaa20_geolocation\':\'https://ladsweb.modaps.eosdis.nasa.gov/missions-and-measurements/products/VJ103IMG\',\n \'cmr\':\'https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html\'}\nPOLICY={\'pilot_id\':\'lk_viirs_observation_evidence_v1\',\'days\':DAYS,\'products\':PRODUCTS,\'sources\':SOURCES,\n \'sampling\':\'purposive schema/outage feasibility sample, not independent model validation\',\n \'standard_only\':True,\'h3_resolution\':8,\'candidate_time_windows_minutes\':[30,60,90],\n \'candidate_pairing\':\'all cross-sensor granule candidates within time windows; not assumed independent or one-to-one\',\n \'support\':\'pixel centres only; no footprint areas, cloud-free area fractions or complete coverage inferred\',\n \'screen\':\'fire-mask classes 5/7/8/9, nominal input QA bits 0–6, residual bowtie bit 22 absent; not an official replacement algorithm\',\n \'scope\':\'all native fire-mask classes, not vegetation-only model target; type attribution not independently available\',\n \'zero_rule\':\'missing catalogue, missing companion, failed download or screen is unavailable, never an observed zero\',\n \'replacement_enabled\':False,\'early_history_bands_enabled\':False,\'coverage_verified\':False,\'frontend_changed\':False}\n\ndef sha(raw):return hashlib.sha256(raw).hexdigest()\ndef acquisition(name):\n    match=re.search(r\'\\.A(\\d{7})\\.(\\d{4})\\.\',name)\n    if not match:raise ValueError(\'Unrecognized acquisition filename\')\n    return \'.\'.join(match.groups())\n\ndef query(params,destination):\n    url=\'https://cmr.earthdata.nasa.gov/search/granules.json?\'+urllib.parse.urlencode(params)\n    receipt=destination.with_suffix(\'.receipt.json\')\n    if destination.exists() and receipt.exists():\n        raw=destination.read_bytes();r=json.loads(receipt.read_text())\n        if r[\'url\']!=url or r[\'sha256\']!=sha(raw):raise ValueError(\'Changed metadata cache\')\n        return json.loads(raw),r\n    for attempt in range(4):\n        try:\n            with urllib.request.urlopen(urllib.request.Request(url,headers={\'User-Agent\':\'CinderTrace-observation-evidence/1.0\'}),timeout=60) as response:\n                raw=response.read();hits=response.headers.get(\'CMR-Hits\')\n            body=json.loads(raw)\n            if not isinstance(body.get(\'feed\',{}).get(\'entry\'),list):raise ValueError(\'Unexpected CMR schema\')\n            r={\'url\':url,\'sha256\':sha(raw),\'bytes\':len(raw),\'cmr_hits\':int(hits) if hits is not None else None,\'retrieved_utc\':datetime.now(timezone.utc).isoformat()}\n            destination.write_bytes(raw);receipt.write_text(json.dumps(r,indent=2)+\'\\n\')\n            return body,r\n        except (urllib.error.URLError,TimeoutError):\n            if attempt==3:raise\n            time.sleep(min(2**attempt,4))\n\ndef normalize(entry,product,day):\n    spec=PRODUCTS[product];name=entry[\'producer_granule_id\']\n    if entry.get(\'collection_concept_id\')!=spec[\'collection\'] or not name.startswith(product+\'.\'):raise ValueError(\'Wrong collection/product\')\n    urls=[l[\'href\'] for l in entry.get(\'links\',[]) if not l.get(\'inherited\') and l.get(\'rel\',\'\').endswith(\'/data#\') and l.get(\'href\',\'\').startswith(\'https://\') and urllib.parse.urlsplit(l[\'href\']).path.endswith(\'.nc\')]\n    if len(urls)!=1:raise ValueError(\'Expected one science NetCDF link\')\n    return {\'id\':entry[\'id\'],\'product\':product,\'satellite\':spec[\'satellite\'],\'kind\':spec[\'kind\'],\n     \'name\':name,\'acquisition\':acquisition(name),\'date\':day,\'time_start\':entry[\'time_start\'],\'time_end\':entry[\'time_end\'],\n     \'day_night\':entry.get(\'day_night_flag\',\'UNKNOWN\'),\'download_url\':urls[0],\n     \'catalogue_size_mb\':float(entry.get(\'granule_size\',0)),\'metadata_entry_sha256\':sha(json.dumps(entry,sort_keys=True).encode()),\n     \'catalogue_footprint_only\':True}\n\ndef join_companions(records):\n    fire=[r for r in records if r[\'kind\']==\'fire\'];result=[]\n    for f in fire:\n        g=[g for g in records if g[\'kind\']==\'geolocation\' and g[\'satellite\']==f[\'satellite\'] and g[\'acquisition\']==f[\'acquisition\']]\n        status=\'unique_catalogue_companion\' if len(g)==1 else \'missing_companion\' if not g else \'ambiguous_companion\'\n        if len(g)==1 and (pd.Timestamp(g[0][\'time_start\'])!=pd.Timestamp(f[\'time_start\']) or pd.Timestamp(g[0][\'time_end\'])!=pd.Timestamp(f[\'time_end\'])):status=\'time_extent_mismatch\'\n        result.append({\'fire_id\':f[\'id\'],\'geo_id\':g[0][\'id\'] if len(g)==1 else None,\'satellite\':f[\'satellite\'],\'date\':f[\'date\'],\'acquisition\':f[\'acquisition\'],\'status\':status})\n    return result\n\ndef time_candidates(records):\n    fire=[r for r in records if r[\'kind\']==\'fire\'];rows=[]\n    for a in fire:\n        if a[\'satellite\']!=\'Suomi-NPP\':continue\n        for b in fire:\n            if b[\'satellite\']!=\'NOAA-20\' or a[\'date\']!=b[\'date\']:continue\n            # Use separation of granule midpoints. Date-boundary candidates are out of this small pilot.\n            am=pd.Timestamp(a[\'time_start\'])+(pd.Timestamp(a[\'time_end\'])-pd.Timestamp(a[\'time_start\']))/2\n            bm=pd.Timestamp(b[\'time_start\'])+(pd.Timestamp(b[\'time_end\'])-pd.Timestamp(b[\'time_start\']))/2\n            dt=abs((am-bm).total_seconds())/60\n            if dt<=90:rows.append({\'snpp_id\':a[\'id\'],\'noaa20_id\':b[\'id\'],\'date\':a[\'date\'],\'midpoint_separation_minutes\':dt,\n             \'within_30\':dt<=30,\'within_60\':dt<=60,\'within_90\':True,\'day_night_agrees\':a[\'day_night\']==b[\'day_night\'] and a[\'day_night\'] in [\'DAY\',\'NIGHT\'],\n             \'status\':\'time_candidate_only_pixel_overlap_unverified\'})\n    return rows\n\ndef discover(boundary_path=None,output=None):\n    out=Path(OUT if output is None else output);out.mkdir(parents=True,exist_ok=True);rawdir=out/\'cmr\';rawdir.mkdir(exist_ok=True)\n    bp=Path(boundary_path or ROOT/\'data/atlas/canonical_lk_2000_2024_v1/region.geojson\');raw=bp.read_bytes()\n    if sha(raw)!=BOUNDARY_SHA:raise ValueError(\'Boundary differs from verified study region\')\n    from shapely.geometry import shape\n    region=shape(json.loads(raw)[\'features\'][0][\'geometry\']);bbox=\',\'.join(map(str,region.bounds));(out/\'region.geojson\').write_bytes(raw)\n    records=[];counts=[]\n    for day in DAYS:\n        for product,spec in PRODUCTS.items():\n            found=[];page=1;expected=None\n            while True:\n                params={\'collection_concept_id\':spec[\'collection\'],\'bounding_box\':bbox,\'temporal\':day+\'T00:00:00Z,\'+day+\'T23:59:59.999Z\',\'page_size\':200,\'page_num\':page}\n                body,receipt=query(params,rawdir/f\'{product}_{day}_{page}.json\');entries=body[\'feed\'][\'entry\']\n                if expected is None:expected=receipt[\'cmr_hits\']\n                elif receipt[\'cmr_hits\']!=expected:raise ValueError(\'Catalogue changed during pagination; retry in a fresh output folder\')\n                found.extend(entries)\n                if len(entries)<200:break\n                page+=1\n                if page>100:raise ValueError(\'Unexpected pilot catalogue size\')\n            if expected is not None and len(found)!=expected:raise ValueError(\'Incomplete CMR pagination\')\n            if len({e[\'id\'] for e in found})!=len(found):raise ValueError(\'Duplicate CMR entries\')\n            # Temporal queries overlap intervals; retain granules whose start falls on the requested UTC day.\n            found=[e for e in found if e[\'time_start\'][:10]==day]\n            normalized=[normalize(e,product,day) for e in found];records.extend(normalized)\n            counts.append({\'date\':day,\'role\':DAYS[day],\'product\':product,\'records\':len(normalized),\'status\':\'catalogue_listed\' if normalized else \'no_catalogue_records_not_zero_observations\'})\n        print(\'Catalogued\',day,flush=True)\n    joins=join_companions(records);candidates=time_candidates(records)\n    report={\'policy\':POLICY,\'created_utc\':datetime.now(timezone.utc).isoformat(),\'boundary_sha256\':sha(raw),\'bbox\':list(region.bounds),\n     \'records\':records,\'daily_catalogue\':counts,\'companions\':joins,\'time_candidates\':candidates,\n     \'catalogue_size_mb\':sum(r[\'catalogue_size_mb\'] for r in records),\n     \'status\':\'metadata_only_science_screen_pending\',\'science_files_verified\':0,\'usable_coverage_verified\':False}\n    (out/\'inventory.json\').write_text(json.dumps(report,indent=2,allow_nan=False)+\'\\n\');(out/\'method.json\').write_text(json.dumps(POLICY,indent=2)+\'\\n\')\n    return report\n\nclass ScienceAccessRequired(RuntimeError):pass\n\ndef download(record,session,cache):\n    """Resumable per file; stores no auth headers, cookies, tokens or passwords."""\n    cache=Path(cache);cache.mkdir(parents=True,exist_ok=True)\n    filename=Path(urllib.parse.urlsplit(record[\'download_url\']).path).name\n    if filename!=record[\'name\']+\'.nc\' and filename!=record[\'name\']:raise ValueError(\'URL/name mismatch\')\n    target=cache/filename;rp=cache/(filename+\'.receipt.json\')\n    if target.exists() and rp.exists():\n        r=json.loads(rp.read_text());raw=target.read_bytes()\n        if r[\'id\']!=record[\'id\'] or r[\'sha256\']!=sha(raw) or r[\'bytes\']!=len(raw):raise ValueError(\'Changed verified science cache\')\n        return target,r\n    part=cache/(filename+\'.part\');reason=\'network\'\n    for attempt in range(4):\n        try:\n            with session.get(record[\'download_url\'],stream=True,timeout=(30,180)) as response:\n                if response.status_code!=200:\n                    reason=\'HTTP \'+str(response.status_code)\n                    if response.status_code in (401,403):break\n                    response.raise_for_status()\n                expected=int(response.headers.get(\'Content-Length\',0));digest=hashlib.sha256();n=0\n                with part.open(\'wb\') as handle:\n                    for chunk in response.iter_content(1024*1024):\n                        if chunk:handle.write(chunk);digest.update(chunk);n+=len(chunk)\n                if expected and n!=expected:raise OSError(\'Incomplete content length\')\n            with part.open(\'rb\') as f:signature=f.read(8)\n            if signature!=b\'\\x89HDF\\r\\n\\x1a\\n\':raise ValueError(\'Response is not NetCDF4/HDF5 science data\')\n            r={\'id\':record[\'id\'],\'product\':record[\'product\'],\'acquisition\':record[\'acquisition\'],\'bytes\':n,\'sha256\':digest.hexdigest(),\n             \'retrieved_utc\':datetime.now(timezone.utc).isoformat(),\'http_status\':200,\'status\':\'content_length_and_signature_verified_not_provider_checksum\'}\n            part.replace(target);rp.write_text(json.dumps(r,indent=2)+\'\\n\');return target,r\n        except Exception as error:\n            # Do not print request exceptions; they can contain provider credential redirects.\n            reason=type(error).__name__\n            if attempt<3:time.sleep(min(2**attempt,4))\n        finally:\n            if part.exists():part.unlink()\n    if reason in [\'HTTP 401\',\'HTTP 403\']:raise ScienceAccessRequired(f\'{filename}: {reason}. Check Earthdata authorization for this provider; no zero written.\')\n    raise RuntimeError(f\'{filename}: {reason}. No zero written. Verified prior files retained; review Earthdata/provider access.\') from None\n\ndef variables(dataset,prefix=\'\'):\n    result={prefix+name:var for name,var in dataset.variables.items()}\n    for name,group in dataset.groups.items():result.update(variables(group,prefix+name+\'/\'))\n    return result\n\ndef find_variable(dataset,name):\n    norm=lambda s:re.sub(\'[^a-z0-9]\',\'\',s.lower())\n    hits=[(path,var) for path,var in variables(dataset).items() if norm(path.split(\'/\')[-1])==norm(name)]\n    if len(hits)!=1:raise ValueError(\'Missing or ambiguous science variable: \'+name)\n    return hits[0]\n\ndef screen_arrays(mask,qa,latitude,longitude,region):\n    """Chunk kernel: preserve every local class; good centres are an explicit screen."""\n    from shapely import intersects_xy\n    m=np.ma.asarray(mask);q=np.ma.asarray(qa);lat=np.ma.asarray(latitude);lon=np.ma.asarray(longitude)\n    if len(m.shape)!=2 or len({m.shape,q.shape,lat.shape,lon.shape})!=1:raise ValueError(\'Fire/geolocation grid mismatch\')\n    lat=np.ma.filled(lat.astype(float),np.nan);lon=np.ma.filled(lon.astype(float),np.nan)\n    # Invalid/masked QA must never be converted to nominal (zero).\n    goodgeo=np.isfinite(lat)&np.isfinite(lon)&(abs(lat)<=90)&(abs(lon)<=180)\n    local=goodgeo&intersects_xy(region,lon,lat)\n    mv=np.ma.filled(m.astype(float),np.nan);qv=np.ma.filled(q.astype(np.uint32),np.uint32(2**32-1))\n    known=~np.ma.getmaskarray(m)&np.isfinite(mv)&np.isin(mv,np.arange(10))\n    if (local&~known).any():raise ValueError(\'Unknown or missing local fire-mask class\')\n    nominal=(qv&127)==0;not_duplicate=(qv&(1<<22))==0\n    clear=local&np.isin(mv,[5,7,8,9])&nominal&not_duplicate&~np.ma.getmaskarray(q)\n    fires=clear&np.isin(mv,[7,8,9])\n    return {\'local\':local,\'mask\':mv,\'qa\':qv,\'qa_valid\':~np.ma.getmaskarray(q),\'clear\':clear,\'fires\':fires,\'latitude\':lat,\'longitude\':lon}\n\ndef screen_file(fire_path,geo_path,record,region):\n    from netCDF4 import Dataset\n    import h3\n    rows=[];summary={\'fire_id\':record[\'id\'],\'product\':record[\'product\'],\'satellite\':record[\'satellite\'],\'date\':record[\'date\'],\n      \'time_start\':record[\'time_start\'],\'time_end\':record[\'time_end\'],\'day_night\':record[\'day_night\'],\n      \'local_geolocated_centres\':0,\'nominal_clear_centres\':0,\'screened_fire_centres\':0,\'local_residual_bowtie_centres\':0,\n      \'scope\':\'all native fire classes; no vegetation type attribution\',\'status\':\'pixel_centre_screen_only_not_complete_coverage\'}\n    with Dataset(fire_path) as fire,Dataset(geo_path) as geo:\n        fm,mask=find_variable(fire,\'fire mask\');fq,qa=find_variable(fire,\'algorithm QA\');gl,lat=find_variable(geo,\'latitude\');go,lon=find_variable(geo,\'longitude\')\n        if len(mask.shape)!=2 or mask.shape!=lat.shape or mask.shape!=lon.shape or mask.shape!=qa.shape:raise ValueError(\'Companion grid shape mismatch\')\n        summary.update({\'fire_mask_variable\':fm,\'qa_variable\':fq,\'latitude_variable\':gl,\'longitude_variable\':go,\'shape\':list(mask.shape)})\n        # Independent sparse-fire coordinates check the zero-based line/sample alignment.\n        _,pl=find_variable(fire,\'FP_line\');_,ps=find_variable(fire,\'FP_sample\');_,plat=find_variable(fire,\'FP_latitude\');_,plon=find_variable(fire,\'FP_longitude\')\n        alignment=[]\n        for r,c,a,b in zip(pl[:100],ps[:100],plat[:100],plon[:100]):\n            r=int(r);c=int(c)\n            if r<0 or c<0 or r>=mask.shape[0] or c>=mask.shape[1]:raise ValueError(\'Invalid zero-based sparse-fire grid index\')\n            x=float(lat[r,c]);y=float(lon[r,c]);delta=max(abs(x-float(a)),abs(((y-float(b)+180)%360)-180))\n            if not np.isfinite(delta) or delta>.01:raise ValueError(\'Sparse-fire geolocation does not align with companion grid\')\n            alignment.append(delta)\n        summary[\'alignment_check_pixels\']=len(alignment);summary[\'alignment_max_error_degrees\']=max(alignment) if alignment else None\n        summary[\'alignment_status\']=\'sparse_fire_coordinates_agree\' if alignment else \'no_sparse_fires_only_catalogue_time_and_shape_checks\'\n        for start in range(0,mask.shape[0],256):\n            sl=slice(start,min(start+256,mask.shape[0]));x=screen_arrays(mask[sl,:],qa[sl,:],lat[sl,:],lon[sl,:],region)\n            summary[\'local_geolocated_centres\']+=int(x[\'local\'].sum());summary[\'nominal_clear_centres\']+=int(x[\'clear\'].sum());summary[\'screened_fire_centres\']+=int(x[\'fires\'].sum())\n            summary[\'local_residual_bowtie_centres\']+=int((x[\'local\']&x[\'qa_valid\']&((x[\'qa\']&(1<<22))!=0)).sum())\n            for cls in range(10):summary[\'class_\'+str(cls)]=summary.get(\'class_\'+str(cls),0)+int((x[\'local\']&(x[\'mask\']==cls)).sum())\n            ix=x[\'local\'];la=x[\'latitude\'][ix];lo=x[\'longitude\'][ix]\n            if not len(la):continue\n            chunk=pd.DataFrame({\'cell\':[h3.latlng_to_cell(float(a),float(b),8) for a,b in zip(la,lo)],\n              \'local_centres\':1,\'clear_centres\':x[\'clear\'][ix].astype(int),\'fire_centres\':x[\'fires\'][ix].astype(int)})\n            rows.append(chunk.groupby(\'cell\',as_index=False).sum())\n    cells=pd.concat(rows).groupby(\'cell\',as_index=False).sum() if rows else pd.DataFrame(columns=[\'cell\',\'local_centres\',\'clear_centres\',\'fire_centres\'])\n    for name in [\'fire_id\',\'satellite\',\'date\',\'time_start\',\'time_end\']:cells[name]=summary[name]\n    # Presence of a centre does not imply coverage of the full H3 cell.\n    return summary,cells\n\ndef candidate_diagnostics(candidates,cells):\n    rows=[]\n    for c in candidates:\n        base=dict(c);a=cells[cells.fire_id.eq(c[\'snpp_id\'])];b=cells[cells.fire_id.eq(c[\'noaa20_id\'])]\n        if not len(a) or not len(b):\n            base.update({\'status\':\'one_or_both_screens_missing_or_no_regional_centres\',\'common_clear_centre_cells\':None,\n              \'snpp_active_cells_on_common_support\':None,\'noaa20_active_cells_on_common_support\':None});rows.append(base);continue\n        # Cell labels approximate spatial co-location; actual pixel footprints differ.\n        common=a[a.clear_centres.gt(0)].merge(b[b.clear_centres.gt(0)],on=\'cell\',suffixes=(\'_snpp\',\'_noaa20\'))\n        base.update({\'status\':\'coarse_cell_centre_overlap_diagnostic_not_matched_pixel_footprints\',\n         \'common_clear_centre_cells\':len(common),\n         \'snpp_active_cells_on_common_support\':int(common.fire_centres_snpp.gt(0).sum()) if len(common) else None,\n         \'noaa20_active_cells_on_common_support\':int(common.fire_centres_noaa20.gt(0).sum()) if len(common) else None})\n        rows.append(base)\n    return pd.DataFrame(rows)\n\ndef daily_screen_status(inventory,summaries):\n    rows=[]\n    for day in DAYS:\n        for satellite in [\'Suomi-NPP\',\'NOAA-20\']:\n            planned=[r for r in inventory[\'records\'] if r[\'kind\']==\'fire\' and r[\'date\']==day and r[\'satellite\']==satellite]\n            done=[r for r in summaries if r[\'date\']==day and r[\'satellite\']==satellite]\n            all_done=bool(planned) and len(done)==len(planned)\n            local=sum(r[\'local_geolocated_centres\'] for r in done)\n            supported=all_done and local>0\n            rows.append({\'date\':day,\'satellite\':satellite,\'catalogue_fire_granules\':len(planned),\'screened_granules\':len(done),\n                \'status\':\'catalogue_absent_not_zero_observations\' if not planned else \'incomplete_science_screen\' if not all_done else \'no_local_centres_not_zero_activity\' if not local else \'catalogued_granules_screened_coverage_unverified\',\n                \'screened_fire_centres\':sum(r[\'screened_fire_centres\'] for r in done) if supported else None,\n                \'nominal_clear_centres\':sum(r[\'nominal_clear_centres\'] for r in done) if supported else None,\n                \'observing_coverage_verified\':False})\n    return pd.DataFrame(rows)\n\ndef run_science(inventory,session,cache,output):\n    from shapely.geometry import shape\n    out=Path(output);out.mkdir(parents=True,exist_ok=True);source=Path(inventory);inv=json.loads(source.read_text());region_raw=(source.parent/\'region.geojson\').read_bytes()\n    if sha(region_raw)!=BOUNDARY_SHA:raise ValueError(\'Boundary changed\')\n    region=shape(json.loads(region_raw)[\'features\'][0][\'geometry\']);lookup={r[\'id\']:r for r in inv[\'records\']};summaries=[];cells=[];receipts=[];failures=[]\n    # Each completed swath is retained immediately, so later access/schema failures do not lose prior work.\n    access_stopped=False\n    for join in inv[\'companions\']:\n        if access_stopped:\n            failures.append({**join,\'status\':\'not_attempted_after_access_denial\'});continue\n        if join[\'status\']!=\'unique_catalogue_companion\':failures.append({**join,\'status\':\'unavailable_companion\'});continue\n        f=lookup[join[\'fire_id\']];g=lookup[join[\'geo_id\']]\n        try:\n            fp,fr=download(f,session,cache);gp,gr=download(g,session,cache);receipts.extend([fr,gr])\n            summary,cell=screen_file(fp,gp,f,region);summaries.append(summary);cells.append(cell)\n            pd.DataFrame(summaries).to_parquet(out/\'granule_screening.parquet\',index=False)\n            pd.concat(cells,ignore_index=True).to_parquet(out/\'cell_centre_support.parquet\',index=False)\n            print(\'Screened\',f[\'name\'],flush=True)\n        except Exception as error:\n            # Error text may include authenticated URLs from a third-party library; retain safe type only.\n            failures.append({**join,\'status\':\'unavailable_science_screen\',\'error_type\':type(error).__name__})\n            if isinstance(error,ScienceAccessRequired):access_stopped=True\n            print(\'Unavailable screen:\',f[\'name\'],type(error).__name__,flush=True)\n        finally:\n            (out/\'download_receipts.json\').write_text(json.dumps(receipts,indent=2)+\'\\n\');(out/\'failures.json\').write_text(json.dumps(failures,indent=2)+\'\\n\')\n    combined=pd.concat(cells,ignore_index=True) if cells else pd.DataFrame(columns=[\'fire_id\',\'cell\',\'clear_centres\',\'fire_centres\'])\n    pd.DataFrame(summaries,columns=None if summaries else [\'fire_id\',\'satellite\',\'date\',\'status\']).to_parquet(out/\'granule_screening.parquet\',index=False)\n    combined.to_parquet(out/\'cell_centre_support.parquet\',index=False)\n    (out/\'download_receipts.json\').write_text(json.dumps(receipts,indent=2)+\'\\n\')\n    (out/\'failures.json\').write_text(json.dumps(failures,indent=2)+\'\\n\')\n    daily=daily_screen_status(inv,summaries)\n    daily.to_parquet(out/\'daily_screen_status.parquet\',index=False)\n    diagnostic=candidate_diagnostics(inv[\'time_candidates\'],combined);diagnostic.to_parquet(out/\'candidate_pair_diagnostics.parquet\',index=False)\n    decision={\'pilot_id\':POLICY[\'pilot_id\'],\'inventory_sha256\':sha(source.read_bytes()),\'planned_fire_granules\':len(inv[\'companions\']),\n     \'screened_fire_granules\':len(summaries),\'failed_or_unavailable\':len(failures),\'access_denial_stopped_downloads\':access_stopped,\'status\':\'exploratory_pixel_centre_evidence_only\',\n     \'zero_months_restored\':0,\'replacement_validated\':False,\'gap_fill_enabled\':False,\'coverage_verified\':False,\n     \'independent_uncertainty_validation_completed\':False,\'frontend_changed\':False,\n     \'limitations\':[\'Small purposive sample; no model tuning or validation scores\',\'Granule times, not exact per-pixel observation times\',\n       \'Coarse cell-centre overlap is not matched physical footprints or full regional area support\',\'Candidate pairs may share granules; do not sum pair counts\',\n       \'No vegetation type attribution: do not compare directly to vegetation-only transfer scores\',\'Missing files and failed screens remain unavailable\']}\n    (out/\'decision.json\').write_text(json.dumps(decision,indent=2)+\'\\n\');(out/\'method.json\').write_text(json.dumps(POLICY,indent=2)+\'\\n\')\n    decision[\'files\']={}\n    for path in sorted(out.iterdir()):\n        if path.is_file() and path.name not in [\'manifest.json\']:\n            raw=path.read_bytes();decision[\'files\'][path.name]={\'bytes\':len(raw),\'sha256\':sha(raw)}\n    (out/\'manifest.json\').write_text(json.dumps(decision,indent=2,allow_nan=False)+\'\\n\')\n    return decision\n\n'
PIXEL_SOURCE='"""Reprocess verified pilot cache without network or credentials.\nNearest pixel centres are diagnostics, not physical footprint equivalence.\n"""\nfrom pathlib import Path\nimport hashlib,json,urllib.parse\nimport numpy as np\nimport pandas as pd\nfrom scipy.spatial import cKDTree\nfrom scipy.optimize import linear_sum_assignment\nfrom . import atlas_observation_evidence as e\n\nPOLICY={\'audit_id\':\'lk_cached_pixel_pass_audit_v1\',\'network_downloads\':False,\n \'pass_grouping\':\'same satellite/UTC date/catalogue day-night; successive intervals separated by at most six minutes; pilot edges may truncate passes\',\n \'pairing\':\'one-to-one maximum-cardinality minimum-midpoint-time assignment separately at 30/60/90 minutes; unknown/mixed day-night excluded\',\n \'matching\':\'mutual nearest nominal-clear pixel centres on sphere; 250/500 metre sensitivity; not footprint intersection or simultaneity\',\n \'view_screens_degrees\':[None,40,60],\'archive_match_metres\':2,\n \'archive_match\':\'same satellite and granule interval; mutual unique coordinate candidate within two metres; preserve ambiguous/unmatched states\',\n \'archive_type\':\'carried from uniquely matched standard observation, not independently validated burning type\',\n \'coverage_verified\':False,\'replacement_validated\':False,\'uncertainty_validated\':False,\'frontend_changed\':False}\nRADIUS=6371008.8\n\ndef require(ok,message):\n    if not ok:raise ValueError(message)\ndef file_sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\ndef xyz(lat,lon):\n    a=np.radians(np.asarray(lat,float));b=np.radians(np.asarray(lon,float))\n    return np.column_stack((np.cos(a)*np.cos(b),np.cos(a)*np.sin(b),np.sin(a)))\ndef chord(metres):return 2*np.sin(metres/(2*RADIUS))\ndef mutual_neighbours(a,b):\n    if not len(a) or not len(b):return np.array([],int),np.array([],int),np.array([],float)\n    aa=xyz(a.latitude,a.longitude);bb=xyz(b.latitude,b.longitude)\n    da,ia=cKDTree(bb).query(aa);_,ib=cKDTree(aa).query(bb)\n    ix=np.flatnonzero(ib[ia]==np.arange(len(aa)));iy=ia[ix]\n    return ix,iy,2*RADIUS*np.arcsin(np.minimum(1,da[ix]/2))\n\ndef archive_match(native,archive):\n    out=native.copy();out[\'archive_match_status\']=\'unmatched\';out[\'archive_observation_id\']=None;out[\'archive_type\']=pd.Series(pd.NA,index=out.index,dtype=\'Int64\');out[\'archive_distance_metres\']=np.nan\n    proposals={};reverse={}\n    for i,row in out.iterrows():\n        start=pd.Timestamp(row.time_start);end=pd.Timestamp(row.time_end)\n        a=archive[(archive.satellite==row.satellite)&(archive.acq_datetime>=start)&(archive.acq_datetime<end)]\n        if not len(a):continue\n        d=np.linalg.norm(xyz(a.latitude,a.longitude)-xyz([row.latitude],[row.longitude])[0],axis=1)\n        candidates=list(a.index[d<=chord(POLICY[\'archive_match_metres\'])]);proposals[i]=candidates\n        for j in candidates:reverse.setdefault(j,[]).append(i)\n    for i,choices in proposals.items():\n        if not choices:continue\n        if len(choices)!=1 or len(reverse[choices[0]])!=1:out.loc[i,\'archive_match_status\']=\'ambiguous\';continue\n        j=choices[0];a=archive.loc[j];out.loc[i,\'archive_match_status\']=\'unique_standard_observation\';out.loc[i,\'archive_observation_id\']=a.observation_id;out.loc[i,\'archive_type\']=int(a.type)\n        delta=np.linalg.norm(xyz([out.loc[i,\'latitude\']],[out.loc[i,\'longitude\']])-xyz([a.latitude],[a.longitude]))\n        out.loc[i,\'archive_distance_metres\']=2*RADIUS*np.arcsin(min(1,delta/2))\n    return out\n\ndef group_passes(records):\n    rows=[];f=pd.DataFrame([r for r in records if r[\'kind\']==\'fire\'])\n    for (sat,date,flag),group in f.groupby([\'satellite\',\'date\',\'day_night\'],sort=True):\n        rank=0;end=None\n        for row in group.sort_values([\'time_start\',\'id\']).to_dict(\'records\'):\n            start=pd.Timestamp(row[\'time_start\']);stop=pd.Timestamp(row[\'time_end\'])\n            if end is None or start>end+pd.Timedelta(minutes=6):rank+=1\n            pid=f\'{sat}_{date}_{flag}_{rank:02d}\'\n            rows.append({\'fire_id\':row[\'id\'],\'pass_id\':pid,\'satellite\':sat,\'date\':date,\'day_night\':flag,\'time_start\':row[\'time_start\'],\'time_end\':row[\'time_end\']})\n            end=max(end,stop) if end is not None else stop\n    return pd.DataFrame(rows)\n\ndef pair_passes(passes,window):\n    matched=[];unmatched=[]\n    for (day,flag),f in passes.groupby([\'date\',\'day_night\'],sort=True):\n        a=f[f.satellite.eq(\'Suomi-NPP\')].sort_values(\'pass_id\');b=f[f.satellite.eq(\'NOAA-20\')].sort_values(\'pass_id\')\n        good=flag in [\'DAY\',\'NIGHT\'];aids=list(a.pass_id);bids=list(b.pass_id);used=set();useda=set()\n        if good and len(a) and len(b):\n            at=np.array([pd.Timestamp(x).timestamp() for x in a.midpoint_utc]);bt=np.array([pd.Timestamp(x).timestamp() for x in b.midpoint_utc])\n            delta=np.abs(at[:,None]-bt[None,:])/60\n            eligible=(delta<=window)&a.complete_science_screen.to_numpy(bool)[:,None]&b.complete_science_screen.to_numpy(bool)[None,:]&a.local_centres.gt(0).to_numpy()[:,None]&b.local_centres.gt(0).to_numpy()[None,:]\n            cost=np.concatenate([np.where(eligible,delta,1e9),np.full((len(a),len(a)),1e6)],axis=1)\n            ai,bi=linear_sum_assignment(cost)\n            for x,y in zip(ai,bi):\n                if y>=len(b) or not eligible[x,y]:continue\n                useda.add(aids[x]);used.add(bids[y]);matched.append({\'window_minutes\':window,\'date\':day,\'day_night\':flag,\'snpp_pass\':aids[x],\'noaa20_pass\':bids[y],\'separation_minutes\':float(delta[x,y])})\n        for ids,usedids in [(aids,useda),(bids,used)]:\n            for pid in ids:\n                if pid not in usedids:unmatched.append({\'window_minutes\':window,\'pass_id\':pid,\'date\':day,\'day_night\':flag,\'status\':\'no_eligible_one_to_one_partner\'})\n    return pd.DataFrame(matched,columns=[\'window_minutes\',\'date\',\'day_night\',\'snpp_pass\',\'noaa20_pass\',\'separation_minutes\']),pd.DataFrame(unmatched)\n\ndef degree_variable(dataset,name,shape):\n    path,var=e.find_variable(dataset,name);require(var.shape==shape,\'Geometry grid mismatch: \'+name)\n    units=str(getattr(var,\'units\',\'\')).lower();require(\'degree\' in units,\'Unrecognized geometry units: \'+name)\n    return path,var\n\ndef extract_granule(fire_path,geo_path,record,region):\n    from netCDF4 import Dataset\n    native=[];support=[];bins=[];summary={\'fire_id\':record[\'id\'],\'satellite\':record[\'satellite\'],\'date\':record[\'date\'],\'time_start\':record[\'time_start\'],\'time_end\':record[\'time_end\'],\'day_night\':record[\'day_night\'],\'local_centres\':0,\'clear_centres\':0,\'raw_fire_centres\':0,\'screened_fire_centres\':0,\'geometry_unavailable_centres\':0}\n    with Dataset(fire_path) as fire,Dataset(geo_path) as geo:\n        _,mask=e.find_variable(fire,\'fire mask\');_,qa=e.find_variable(fire,\'algorithm QA\');_,lat=e.find_variable(geo,\'latitude\');_,lon=e.find_variable(geo,\'longitude\')\n        require(mask.shape==qa.shape==lat.shape==lon.shape,\'Native grid mismatch\')\n        _,view=degree_variable(geo,\'sensor_zenith\',mask.shape);_,solar=degree_variable(geo,\'solar_zenith\',mask.shape)\n        # Sparse arrays are independently checked against every local fire-mask location.\n        sparse={}\n        for name in [\'FP_line\',\'FP_sample\',\'FP_latitude\',\'FP_longitude\',\'FP_confidence\',\'FP_ViewZenAng\',\'FP_SolZenAng\']:\n            _,var=e.find_variable(fire,name);sparse[name]=np.ma.asarray(var[:])\n        lengths={len(v) for v in sparse.values()};require(len(lengths)==1,\'Sparse-array length mismatch\')\n        byindex={}\n        for k,(r,c) in enumerate(zip(sparse[\'FP_line\'],sparse[\'FP_sample\'])):\n            key=(int(r),int(c));require(key not in byindex,\'Duplicate sparse fire index\');byindex[key]=k\n        seen=set()\n        for start in range(0,mask.shape[0],256):\n            sl=slice(start,min(start+256,mask.shape[0]));x=e.screen_arrays(mask[sl,:],qa[sl,:],lat[sl,:],lon[sl,:],region)\n            v=np.ma.filled(np.ma.asarray(view[sl,:]).astype(float),np.nan);s=np.ma.filled(np.ma.asarray(solar[sl,:]).astype(float),np.nan)\n            valid=np.isfinite(v)&np.isfinite(s)&(v>=0)&(v<=90)&(s>=0)&(s<=180)\n            ix=x[\'local\'];summary[\'local_centres\']+=int(ix.sum());summary[\'clear_centres\']+=int(x[\'clear\'].sum());summary[\'screened_fire_centres\']+=int(x[\'fires\'].sum());summary[\'geometry_unavailable_centres\']+=int((ix&~valid).sum())\n            raw=ix&np.isin(x[\'mask\'],[7,8,9]);summary[\'raw_fire_centres\']+=int(raw.sum())\n            for r,c in zip(*np.where(raw)):\n                key=(start+int(r),int(c));require(key in byindex,\'Fire mask location absent from sparse arrays\');k=byindex[key];seen.add(key)\n                delta=max(abs(float(sparse[\'FP_latitude\'][k])-x[\'latitude\'][r,c]),abs(float(sparse[\'FP_longitude\'][k])-x[\'longitude\'][r,c]))\n                require(np.isfinite(delta) and delta<=.01,\'Local sparse geolocation mismatch\')\n                require(int(sparse[\'FP_confidence\'][k])==int(x[\'mask\'][r,c]),\'Sparse confidence differs from mask\')\n                bits=int(x[\'qa\'][r,c]);goodqa=bool(x[\'qa_valid\'][r,c]);native.append({\'fire_id\':record[\'id\'],\'satellite\':record[\'satellite\'],\'date\':record[\'date\'],\'time_start\':record[\'time_start\'],\'time_end\':record[\'time_end\'],\n                 \'line\':key[0],\'sample\':key[1],\'latitude\':float(sparse[\'FP_latitude\'][k]),\'longitude\':float(sparse[\'FP_longitude\'][k]),\'confidence_class\':int(x[\'mask\'][r,c]),\'qa\':bits,\'qa_valid\':goodqa,\'geolocation_latitude\':float(x[\'latitude\'][r,c]),\'geolocation_longitude\':float(x[\'longitude\'][r,c]),\'coordinate_difference_degrees\':float(delta),\n                 \'screen_exclusion_reason\':\',\'.join((["masked_qa"] if not goodqa else [])+(["non_nominal_input"] if goodqa and bits&127 else [])+(["residual_bowtie"] if goodqa and bits&(1<<22) else [])) or \'retained\',\n                 \'input_quality_nominal\':goodqa and (bits&127)==0,\'residual_bowtie\':bool(bits&(1<<22)) if goodqa else None,\'screened_fire\':bool(x[\'fires\'][r,c]),\n                 \'sensor_zenith_degrees\':float(v[r,c]) if valid[r,c] else np.nan,\'solar_zenith_degrees\':float(s[r,c]) if valid[r,c] else np.nan,\n                 \'sparse_view_zenith_degrees\':float(sparse[\'FP_ViewZenAng\'][k]),\'sparse_solar_zenith_degrees\':float(sparse[\'FP_SolZenAng\'][k]),\'time_basis\':\'granule_interval_not_pixel_time\'})\n            # Retain all local nominal-clear centres, including unknown geometry,\n            # in derived cache; pair screens explicitly require known geometry.\n            take=x[\'clear\'];rr,cc=np.where(take)\n            if len(rr):support.append(pd.DataFrame({\'latitude\':x[\'latitude\'][take].astype(\'float32\'),\'longitude\':x[\'longitude\'][take].astype(\'float32\'),\n              \'line\':(rr+start).astype(\'int32\'),\'sample\':cc.astype(\'int16\'),\'fire\':x[\'fires\'][take],\n              \'sensor_zenith_degrees\':np.where(valid[take],v[take],np.nan).astype(\'float32\'),\'solar_zenith_degrees\':np.where(valid[take],s[take],np.nan).astype(\'float32\')}))\n            for label,lo,hi in [(\'0_20\',0,20),(\'20_40\',20,40),(\'40_60\',40,60),(\'60_90\',60,91)]:\n                selected=ix&valid&(v>=lo)&(v<hi);bins.append({\'fire_id\':record[\'id\'],\'view_bin\':label,\'local_centres\':int(selected.sum()),\'clear_centres\':int((selected&x[\'clear\']).sum()),\'screened_fire_centres\':int((selected&x[\'fires\']).sum())})\n        # Check every sparse pixel inside the exact study geometry is represented.\n        from shapely import intersects_xy\n        inside=intersects_xy(region,np.asarray(sparse[\'FP_longitude\'],float),np.asarray(sparse[\'FP_latitude\'],float))\n        expected={(int(sparse[\'FP_line\'][k]),int(sparse[\'FP_sample\'][k])) for k in np.flatnonzero(inside)}\n        require(seen==expected,\'Local sparse/mask fire sets differ\')\n    columns=[\'fire_id\',\'satellite\',\'date\',\'time_start\',\'time_end\',\'line\',\'sample\',\'latitude\',\'longitude\',\'confidence_class\',\'qa\',\'qa_valid\',\'geolocation_latitude\',\'geolocation_longitude\',\'coordinate_difference_degrees\',\'screen_exclusion_reason\',\'input_quality_nominal\',\'residual_bowtie\',\'screened_fire\',\'sensor_zenith_degrees\',\'solar_zenith_degrees\',\'sparse_view_zenith_degrees\',\'sparse_solar_zenith_degrees\',\'time_basis\']\n    n=pd.DataFrame(native,columns=columns);p=pd.concat(support,ignore_index=True) if support else pd.DataFrame(columns=[\'latitude\',\'longitude\',\'line\',\'sample\',\'fire\',\'sensor_zenith_degrees\',\'solar_zenith_degrees\'])\n    b=pd.DataFrame(bins).groupby([\'fire_id\',\'view_bin\'],as_index=False).sum() if bins else pd.DataFrame()\n    return summary,n,p,b\n\ndef pair_diagnostics(a,b):\n    results=[]\n    for cap in POLICY[\'view_screens_degrees\']:\n        def screen(f):\n            ok=f.sensor_zenith_degrees.notna()&f.solar_zenith_degrees.notna()\n            if cap is not None:ok&=f.sensor_zenith_degrees.le(cap)\n            return f[ok].reset_index(drop=True)\n        aa=screen(a);bb=screen(b);ix,iy,d=mutual_neighbours(aa,bb)\n        for radius in [250,500]:\n            keep=d<=radius;matched=int(keep.sum());row={\'view_cap_degrees\':cap,\'distance_cap_metres\':radius,\'snpp_eligible_clear_centres\':len(aa),\'noaa20_eligible_clear_centres\':len(bb),\n             \'mutual_nearest_clear_centre_pairs\':matched,\'snpp_fire_centres_on_support\':int(aa.iloc[ix[keep]].fire.sum()) if matched else None,\n             \'noaa20_fire_centres_on_support\':int(bb.iloc[iy[keep]].fire.sum()) if matched else None,\n             \'median_separation_metres\':float(np.median(d[keep])) if matched else None,\'status\':\'point_centre_diagnostic_not_physical_footprint_match\'}\n            results.append(row)\n    return pd.DataFrame(results)\n\ndef cached_science_files(inv,receipts,cache):\n    cache=Path(cache);rmap={r[\'id\']:r for r in receipts};paths={}\n    for index,r in enumerate(inv[\'records\'],1):\n        require(r[\'id\'] in rmap,\'Missing expected science receipt\');receipt=rmap[r[\'id\']];name=Path(urllib.parse.urlsplit(r[\'download_url\']).path).name\n        require(name==r[\'name\']+\'.nc\' or name==r[\'name\'],\'Cache filename mismatch\');path=cache/name\n        require(path.exists(),\'Missing cached file: \'+name+\'; locate the original science_cache, no download attempted\')\n        require(path.stat().st_size==receipt[\'bytes\'] and file_sha(path)==receipt[\'sha256\'],\'Cached science file differs: \'+name)\n        paths[r[\'id\']]=path\n        if index%8==0 or index==len(inv[\'records\']):print(\'Verified cached science files\',index,\'/\',len(inv[\'records\']),flush=True)\n    return paths\n\ndef run(inv,receipts,boundary,archive,previous_granules,cache,output,derived,method_hash):\n    from shapely.geometry import shape\n    require(e.sha(boundary)==e.BOUNDARY_SHA,\'Boundary changed\');region=shape(json.loads(boundary)[\'features\'][0][\'geometry\'])\n    paths=cached_science_files(inv,receipts,cache);output=Path(output);output.mkdir(parents=True,exist_ok=True);derived=Path(derived);derived.mkdir(parents=True,exist_ok=True)\n    lookup={r[\'id\']:r for r in inv[\'records\']};rmap={r[\'id\']:r for r in receipts};summaries=[];native=[];bins=[];failures=[];support_paths={}\n    for join in inv[\'companions\']:\n        f=lookup[join[\'fire_id\']];g=lookup[join[\'geo_id\']];key=e.sha(f[\'id\'].encode())[:20];folder=derived/key;folder.mkdir(exist_ok=True)\n        signature={\'method_sha256\':method_hash,\'fire_sha256\':rmap[f[\'id\']][\'sha256\'],\'geo_sha256\':rmap[g[\'id\']][\'sha256\']};checkpoint=folder/\'checkpoint.json\'\n        try:\n            if checkpoint.exists():\n                old=json.loads(checkpoint.read_text());require(old[\'signature\']==signature,\'Derived cache method/input changed; use a fresh derived folder\')\n                for name,h in old[\'hashes\'].items():require(file_sha(folder/name)==h,\'Derived cache changed\')\n                summary=json.loads((folder/\'summary.json\').read_text());n=pd.read_parquet(folder/\'native.parquet\');b=pd.read_parquet(folder/\'bins.parquet\')\n            else:\n                summary,n,p,b=extract_granule(paths[f[\'id\']],paths[g[\'id\']],f,region)\n                n.to_parquet(folder/\'native.parquet\',index=False);p.to_parquet(folder/\'clear_centres.parquet\',index=False);b.to_parquet(folder/\'bins.parquet\',index=False);(folder/\'summary.json\').write_text(json.dumps(summary,indent=2)+\'\\n\')\n                checkpoint.write_text(json.dumps({\'signature\':signature,\'hashes\':{name:file_sha(folder/name) for name in [\'native.parquet\',\'clear_centres.parquet\',\'bins.parquet\',\'summary.json\']}},indent=2)+\'\\n\')\n            previous=previous_granules[previous_granules.fire_id.eq(f[\'id\'])].iloc[0]\n            require(summary[\'local_centres\']==previous.local_geolocated_centres and summary[\'clear_centres\']==previous.nominal_clear_centres and summary[\'screened_fire_centres\']==previous.screened_fire_centres,\'Reprocessed screening differs from completed pilot\')\n            require(summary[\'raw_fire_centres\']==previous[[\'class_7\',\'class_8\',\'class_9\']].sum(),\'Reprocessed native count differs\')\n            summaries.append(summary);native.append(n);bins.append(b);support_paths[f[\'id\']]=folder/\'clear_centres.parquet\';print(\'Reprocessed\',f[\'name\'],flush=True)\n        except Exception as error:\n            failures.append({\'fire_id\':f[\'id\'],\'error_type\':type(error).__name__,\'reason\':str(error)[:240]});print(\'Unavailable cached audit:\',f[\'name\'],type(error).__name__,flush=True)\n        (output/\'failures.json\').write_text(json.dumps(failures,indent=2)+\'\\n\')\n    members=group_passes(inv[\'records\']);summary=pd.DataFrame(summaries);pass_rows=[]\n    for pid,group in members.groupby(\'pass_id\',sort=True):\n        local=summary[summary.fire_id.isin(group.fire_id)] if len(summary) else pd.DataFrame();start=min(pd.Timestamp(x) for x in group.time_start);end=max(pd.Timestamp(x) for x in group.time_end)\n        pass_rows.append({\'pass_id\':pid,\'satellite\':group.iloc[0].satellite,\'date\':group.iloc[0].date,\'day_night\':group.iloc[0].day_night,\'time_start\':start.isoformat(),\'time_end\':end.isoformat(),\'midpoint_utc\':(start+(end-start)/2).isoformat(),\n         \'granules\':len(group),\'complete_science_screen\':len(local)==len(group),\'local_centres\':int(local.local_centres.sum()) if len(local)==len(group) else None,\'nominal_clear_centres\':int(local.clear_centres.sum()) if len(local)==len(group) and local.local_centres.sum()>0 else None,\'native_fire_centres\':int(local.raw_fire_centres.sum()) if len(local)==len(group) and local.local_centres.sum()>0 else None,\'screened_fire_centres\':int(local.screened_fire_centres.sum()) if len(local)==len(group) and local.local_centres.sum()>0 else None})\n    passes=pd.DataFrame(pass_rows);matched=[];unmatched=[];diagnostics=[];done_pairs={}\n    def pass_support(pid):\n        files=[support_paths[i] for i in members[members.pass_id.eq(pid)].fire_id if i in support_paths]\n        return pd.concat([pd.read_parquet(p) for p in files],ignore_index=True) if files else pd.DataFrame(columns=[\'latitude\',\'longitude\',\'fire\',\'sensor_zenith_degrees\',\'solar_zenith_degrees\'])\n    for window in [30,60,90]:\n        pairs,un=pair_passes(passes,window);matched.append(pairs);unmatched.append(un)\n        for row in pairs.to_dict(\'records\'):\n            key=(row[\'snpp_pass\'],row[\'noaa20_pass\'])\n            if key not in done_pairs:done_pairs[key]=pair_diagnostics(pass_support(key[0]),pass_support(key[1]));print(\'Compared pass centres\',key,flush=True)\n            d=done_pairs[key].copy()\n            for name,value in row.items():d[name]=value\n            diagnostics.append(d)\n    native_frame=pd.concat(native,ignore_index=True) if native else pd.DataFrame(columns=[\'fire_id\',\'satellite\',\'latitude\',\'longitude\',\'time_start\',\'time_end\'])\n    typed=archive_match(native_frame,archive).merge(members[[\'fire_id\',\'pass_id\']],on=\'fire_id\',how=\'left\',validate=\'many_to_one\');frames={\'native_fire_pixel_audit\':typed,\'granule_audit\':summary,\'view_geometry_bins\':pd.concat(bins,ignore_index=True) if bins else pd.DataFrame(),\n     \'pass_membership\':members,\'pass_inventory\':passes,\'matched_passes\':pd.concat(matched,ignore_index=True),\'unmatched_passes\':pd.concat(unmatched,ignore_index=True),\n     \'mutual_centre_diagnostics\':pd.concat(diagnostics,ignore_index=True) if diagnostics else pd.DataFrame()}\n    for name,frame in frames.items():frame.to_parquet(output/(name+\'.parquet\'),index=False)\n    unmatched_archive=archive[~archive.observation_id.isin(typed.archive_observation_id.dropna())];unmatched_archive.to_parquet(output/\'unmatched_standard_observations.parquet\',index=False)\n    report={\'audit_id\':POLICY[\'audit_id\'],\'input_science_files\':len(paths),\'granules_reprocessed\':len(summary),\'granules_failed\':len(failures),\'native_regional_fire_pixels\':len(typed),\n      \'unique_archive_matches\':int(typed.archive_match_status.eq(\'unique_standard_observation\').sum()),\'ambiguous_archive_matches\':int(typed.archive_match_status.eq(\'ambiguous\').sum()),\'unmatched_standard_observations\':len(unmatched_archive),\n      \'status\':\'exploratory_pass_and_point_audit_only\',\'coverage_verified\':False,\'replacement_validated\':False,\'uncertainty_validated\':False,\'frontend_changed\':False,\'network_downloads\':0,\n      \'limitations\':[\'Only six purposively sampled dates\',\'Point-centre proximity is not physical footprint coverage\',\'Granule intervals, not exact pixel observation times\',\'Type attribution carried from standard archive, not independent fire truth\',\'Window/view/distance sensitivities are not independent validation samples\']}\n    (output/\'decision.json\').write_text(json.dumps(report,indent=2)+\'\\n\');(output/\'method.json\').write_text(json.dumps(POLICY,indent=2)+\'\\n\')\n    (output/\'manifest.json\').write_text(json.dumps({\'decision\':report,\'files\':{p.name:{\'bytes\':p.stat().st_size,\'sha256\':file_sha(p)} for p in sorted(output.iterdir()) if p.is_file() and p.name!=\'manifest.json\'}},indent=2)+\'\\n\')\n    return report\n'
STUDY_SOURCE='"""Resumable expanded VIIRS study. Temporary raw staging; no production exports.\nShared-reference support is diagnostic only. Outage predictors use NOAA-20 alone.\n"""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport json,math,tempfile,urllib.parse\nimport numpy as np\nimport pandas as pd\nimport h3\nfrom scipy.optimize import linear_sum_assignment,nnls\nfrom . import atlas_observation_evidence as e\nfrom . import atlas_cached_pixel_audit as p\n\nPOLICY={\'study_id\':\'lk_viirs_expanded_processing_v1\',\'target\':\'screened vegetation H3-resolution-8 active-cell-days observed by S-NPP in the fixed Sri Lanka polygon\',\n \'scope\':\'regional observed-activity proxy, not true fires, complete coverage, monthly/annual estimates or custom-area calibration\',\n \'partitions\':{\'fit\':[2018,2019,2020],\'calibration\':[2021,2022],\'evaluation\':[2023,2024]},\n \'candidate_models\':[\'global_ratio\',\'seasonal_ratio\',\'positive_affine\'],\'model_selection\':\'fit years only, leave one whole fit year out, WAPE; deterministic tie order\',\n \'primary_predictor\':\'NOAA-20 screened vegetation active cells; calendar month for seasonal candidate; no reference masks/counts/geometry in prediction inputs\',\n \'reference_scope\':\'Unique same-platform coordinate/time match to frozen standard archive supplies type. Ambiguous/unmatched/confidence-disagreeing records withhold vegetation totals.\',\n \'shared_support\':\'all temporal candidates get mutual-clear-centre diagnostics first; geometry admissibility before one-to-one maximum-cardinality minimum-time assignment; no fire counts used for pairing\',\n \'pair_sensitivities\':{\'minutes\':[30,60,90],\'view_cap_degrees\':[None,40,60],\'distance_metres\':[250,500],\'minimum_mutual_centres\':1},\n \'sampling_limits\':\'54 purposive calendar-stratified dates; earlier aggregate scores/pilot inspected; not blind external validation\',\n \'uncertainty\':\'separate calibration dates, scaled absolute residuals; minimum 12 dates, finite order statistic; exploratory date-level 80/90 intervals, no nominal guarantee under temporal dependence\',\n \'temporary_files\':\'Only new raw copies in a dedicated temporary directory are removed. Original pilot and derived caches are preserved.\',\n \'coverage_verified\':False,\'replacement_validated\':False,\'production_export_enabled\':False,\'frontend_changed\':False}\nFEATURE_COLUMNS=[\'date\',\'partition\',\'month_number\',\'noaa20_input_available\',\'noaa20_screened_vegetation_active_cells\',\'noaa20_native_pixels\',\'noaa20_nominal_clear_centres\',\'noaa20_status\']\nLABEL_COLUMNS=[\'date\',\'reference_available\',\'snpp_screened_vegetation_active_cells\',\'snpp_status\']\nNATIVE_COLUMNS=[\'fire_id\',\'satellite\',\'date\',\'time_start\',\'time_end\',\'line\',\'sample\',\'latitude\',\'longitude\',\'confidence_class\',\'screened_fire\',\'sensor_zenith_degrees\']\n\ndef write_json(path,value):\n    path=Path(path);temporary=path.with_name(path.name+\'.tmp\')\n    temporary.write_text(json.dumps(value,indent=2,allow_nan=False)+\'\\n\');temporary.replace(path)\ndef records(frame):return json.loads(frame.to_json(orient=\'records\',date_format=\'iso\'))\ndef checkpoint_valid(folder,name,signature):\n    path=folder/name\n    if not path.exists():return False\n    doc=json.loads(path.read_text());p.require(doc[\'signature\']==signature,\'Checkpoint inputs/method changed; use a fresh run folder\')\n    for file,receipt in doc[\'files\'].items():\n        target=folder/file;p.require(target.is_file() and target.stat().st_size==receipt[\'bytes\'] and p.file_sha(target)==receipt[\'sha256\'],\'Changed checkpoint: \'+file)\n    return True\n\ndef checkpoint(folder,name,signature,files,**extra):\n    doc={\'signature\':signature,\'files\':{file:{\'bytes\':(folder/file).stat().st_size,\'sha256\':p.file_sha(folder/file)} for file in files},**extra}\n    write_json(folder/name,doc)\n\ndef cache_or_download(record,session,original_cache,temporary):\n    name=Path(urllib.parse.urlsplit(record[\'download_url\']).path).name\n    original=Path(original_cache)/name;receipt_path=Path(original_cache)/(name+\'.receipt.json\')\n    if original.exists():\n        p.require(receipt_path.exists(),\'Original cached file has no receipt; review it before reuse\')\n        r=json.loads(receipt_path.read_text());p.require(r[\'id\']==record[\'id\'] and original.stat().st_size==r[\'bytes\'] and p.file_sha(original)==r[\'sha256\'],\'Original cache integrity mismatch\')\n        return original,{**r,\'reused_original_cache\':True}\n    path,receipt=e.download(record,session,temporary)\n    p.require(abs(receipt[\'bytes\']-record[\'catalogue_size_mb\']*1024**2)<2,\'Downloaded size differs from frozen catalogue\')\n    return path,{**receipt,\'reused_original_cache\':False}\n\ndef process_granule(join,lookup,session,original_cache,scratch,derived,method_hash,region):\n    f=lookup[join[\'fire_id\']];g=lookup[join[\'geo_id\']];folder=Path(derived)/e.sha(f[\'id\'].encode())[:24];folder.mkdir(parents=True,exist_ok=True)\n    signature={\'method_hash\':method_hash,\'fire_metadata\':f[\'metadata_entry_sha256\'],\'geo_metadata\':g[\'metadata_entry_sha256\']}\n    if not checkpoint_valid(folder,\'checkpoint.json\',signature):\n        # This context owns only the new transient copies, never user caches.\n        with tempfile.TemporaryDirectory(prefix=\'cindertrace-raw-\',dir=scratch) as tmp:\n            fp,fr=cache_or_download(f,session,original_cache,tmp);gp,gr=cache_or_download(g,session,original_cache,tmp)\n            summary,native,clear,bins=p.extract_granule(fp,gp,f,region)\n            native.to_parquet(folder/\'native.parquet\',index=False);clear.to_parquet(folder/\'clear.parquet\',index=False);bins.to_parquet(folder/\'bins.parquet\',index=False)\n            write_json(folder/\'summary.json\',summary);write_json(folder/\'receipts.json\',[fr,gr])\n            checkpoint(folder,\'checkpoint.json\',signature,[\'native.parquet\',\'clear.parquet\',\'bins.parquet\',\'summary.json\',\'receipts.json\'])\n    return json.loads((folder/\'summary.json\').read_text()),pd.read_parquet(folder/\'native.parquet\'),folder/\'clear.parquet\',json.loads((folder/\'receipts.json\').read_text())\n\ndef attributed_native(native,archive):\n    typed=p.archive_match(native,archive)\n    confidence=archive.set_index(\'observation_id\').confidence\n    typed[\'archive_confidence\']=typed.archive_observation_id.map(confidence)\n    expected=typed.confidence_class.map({7:\'l\',8:\'n\',9:\'h\'}) if \'confidence_class\' in typed else pd.Series(index=typed.index,dtype=\'str\')\n    typed[\'confidence_agrees\']=typed.archive_confidence.eq(expected)&typed.archive_match_status.eq(\'unique_standard_observation\')\n    if len(typed):typed[\'cell\']=[h3.latlng_to_cell(float(a),float(b),8) for a,b in zip(typed.latitude,typed.longitude)]\n    else:typed[\'cell\']=pd.Series(dtype=\'str\')\n    return typed\n\ndef sensor_daily_state(satellite,day,inventory,summaries,typed,archive):\n    planned=[r for r in inventory[\'records\'] if r[\'kind\']==\'fire\' and r[\'date\']==day and r[\'satellite\']==satellite]\n    done=[r for r in summaries if r[\'satellite\']==satellite];n=typed[typed.satellite.eq(satellite)];a=archive[archive.satellite.eq(satellite)]\n    if not planned:return {\'available\':False,\'status\':\'catalogue_absent_not_zero_activity\',\'activity\':None,\'native_pixels\':None,\'clear_centres\':None}\n    if len(done)!=len(planned):return {\'available\':False,\'status\':\'incomplete_science_processing\',\'activity\':None,\'native_pixels\':None,\'clear_centres\':None}\n    complete_match=len(n)==len(a) and n.archive_match_status.eq(\'unique_standard_observation\').all() and n.confidence_agrees.all() and n.archive_observation_id.nunique()==len(a)\n    if not complete_match:return {\'available\':False,\'status\':\'archive_reconciliation_or_type_unavailable\',\'activity\':None,\'native_pixels\':len(n),\'clear_centres\':sum(r[\'clear_centres\'] for r in done)}\n    local=sum(r[\'local_centres\'] for r in done);clear=sum(r[\'clear_centres\'] for r in done)\n    if not local or not clear:return {\'available\':False,\'status\':\'no_nominal_clear_support_not_zero_activity\',\'activity\':None,\'native_pixels\':len(n),\'clear_centres\':clear}\n    count=n[n.screened_fire&n.archive_type.eq(0)].cell.nunique()\n    return {\'available\':True,\'status\':\'recorded_screened_activity_coverage_unverified\',\'activity\':int(count),\'native_pixels\':len(n),\'clear_centres\':clear}\n\ndef daily_tables(slot,inventory,summaries,typed,archive):\n    # Compute each sensor independently. Reference failures cannot change features.\n    day=slot[\'date\'];x=sensor_daily_state(\'NOAA-20\',day,inventory,summaries,typed,archive);y=sensor_daily_state(\'Suomi-NPP\',day,inventory,summaries,typed,archive)\n    feature={\'date\':day,\'partition\':slot[\'partition\'],\'month_number\':int(day[5:7]),\'noaa20_input_available\':x[\'available\'],\n     \'noaa20_screened_vegetation_active_cells\':x[\'activity\'],\'noaa20_native_pixels\':x[\'native_pixels\'],\'noaa20_nominal_clear_centres\':x[\'clear_centres\'],\'noaa20_status\':x[\'status\']}\n    label={\'date\':day,\'reference_available\':y[\'available\'],\'snpp_screened_vegetation_active_cells\':y[\'activity\'],\'snpp_status\':y[\'status\']}\n    return pd.DataFrame([feature],columns=FEATURE_COLUMNS),pd.DataFrame([label],columns=LABEL_COLUMNS)\n\ndef pass_inventory(membership,summaries):\n    s=pd.DataFrame(summaries);rows=[]\n    for pid,f in membership.groupby(\'pass_id\',sort=True):\n        part=s[s.fire_id.isin(f.fire_id)] if len(s) else pd.DataFrame();complete=len(part)==len(f);start=min(pd.Timestamp(t) for t in f.time_start);end=max(pd.Timestamp(t) for t in f.time_end)\n        rows.append({\'pass_id\':pid,\'satellite\':f.iloc[0].satellite,\'date\':f.iloc[0].date,\'day_night\':f.iloc[0].day_night,\'midpoint_utc\':(start+(end-start)/2).isoformat(),\n          \'complete_science_screen\':complete,\'local_centres\':int(part.local_centres.sum()) if complete else None,\'granules\':len(f)})\n    return pd.DataFrame(rows)\n\ndef select_geometry_pairs(passes,diagnostics,window,view,distance):\n    selected=[]\n    diag=diagnostics[diagnostics.distance_cap_metres.eq(distance)&(diagnostics.view_cap_degrees.isna() if view is None else diagnostics.view_cap_degrees.eq(view))]\n    for (date,flag),group in passes.groupby([\'date\',\'day_night\'],sort=True):\n        if flag not in [\'DAY\',\'NIGHT\']:continue\n        a=group[group.satellite.eq(\'Suomi-NPP\')].sort_values(\'pass_id\');b=group[group.satellite.eq(\'NOAA-20\')].sort_values(\'pass_id\')\n        if not len(a) or not len(b):continue\n        lookup={(r.snpp_pass,r.noaa20_pass):r for r in diag.itertuples()};cost=np.full((len(a),len(b)+len(a)),1e6);cost[:,:len(b)]=1e9\n        for ix,ar in enumerate(a.itertuples()):\n            for iy,br in enumerate(b.itertuples()):\n                row=lookup.get((ar.pass_id,br.pass_id))\n                if row is not None and ar.complete_science_screen and br.complete_science_screen and row.separation_minutes<=window and row.mutual_nearest_clear_centre_pairs>=1:cost[ix,iy]=row.separation_minutes\n        xs,ys=linear_sum_assignment(cost)\n        for x,y in zip(xs,ys):\n            if y>=len(b) or cost[x,y]>=1e6:continue\n            selected.append({\'date\':date,\'day_night\':flag,\'window_minutes\':window,\'view_cap_degrees\':view,\'distance_cap_metres\':distance,\n              \'snpp_pass\':a.iloc[x].pass_id,\'noaa20_pass\':b.iloc[y].pass_id,\'separation_minutes\':float(cost[x,y]),\'selection_basis\':\'geometry_admissible_then_time_only_no_fire_counts\'})\n    return pd.DataFrame(selected,columns=[\'date\',\'day_night\',\'window_minutes\',\'view_cap_degrees\',\'distance_cap_metres\',\'snpp_pass\',\'noaa20_pass\',\'separation_minutes\',\'selection_basis\'])\n\ndef geometry_audit(membership,passes,clear_paths):\n    rows=[]\n    def support(pid):\n        return pd.concat([pd.read_parquet(clear_paths[id]) for id in membership[membership.pass_id.eq(pid)].fire_id],ignore_index=True)\n    for ar in passes[passes.satellite.eq(\'Suomi-NPP\')].itertuples():\n        if not ar.complete_science_screen or not ar.local_centres or ar.day_night not in [\'DAY\',\'NIGHT\']:continue\n        for br in passes[passes.satellite.eq(\'NOAA-20\')&passes.date.eq(ar.date)&passes.day_night.eq(ar.day_night)].itertuples():\n            if not br.complete_science_screen or not br.local_centres:continue\n            delta=abs((pd.Timestamp(ar.midpoint_utc)-pd.Timestamp(br.midpoint_utc)).total_seconds())/60\n            if delta>90:continue\n            d=p.pair_diagnostics(support(ar.pass_id),support(br.pass_id));d[\'date\']=ar.date;d[\'day_night\']=ar.day_night;d[\'snpp_pass\']=ar.pass_id;d[\'noaa20_pass\']=br.pass_id;d[\'separation_minutes\']=delta;rows.append(d)\n    cols=[\'view_cap_degrees\',\'distance_cap_metres\',\'mutual_nearest_clear_centre_pairs\',\'separation_minutes\',\'snpp_pass\',\'noaa20_pass\']\n    diagnostics=pd.concat(rows,ignore_index=True) if rows else pd.DataFrame(columns=cols)\n    pairs=[]\n    for window in [30,60,90]:\n        for view in [None,40,60]:\n            for distance in [250,500]:pairs.append(select_geometry_pairs(passes,diagnostics,window,view,distance))\n    return diagnostics,pd.concat(pairs,ignore_index=True)\n\ndef process_date(slot,inputs,session,original_cache,scratch,work,method_hash):\n    from shapely.geometry import shape\n    day=slot[\'date\'];inventory=inputs[\'inventory\'];folder=Path(work)/\'dates\'/day;folder.mkdir(parents=True,exist_ok=True)\n    signature={\'method_hash\':method_hash,\'date\':day}\n    if checkpoint_valid(folder,\'checkpoint.json\',signature):\n        old=json.loads((folder/\'checkpoint.json\').read_text())\n        if old[\'all_planned_granules_processed\']:return old\n        # Withdraw the prior partial commit before replacing its derived tables.\n        (folder/\'checkpoint.json\').unlink()\n    region=shape(json.loads(inputs[\'boundary\'])[\'features\'][0][\'geometry\']);lookup={r[\'id\']:r for r in inventory[\'records\']}\n    joins=[j for j in inventory[\'companions\'] if j[\'date\']==day];summaries=[];native=[];clear_paths={};receipts=[];failures=[];access_denied=False\n    derived=Path(work)/\'derived\';derived.mkdir(exist_ok=True)\n    for join in joins:\n        try:\n            p.require(join[\'status\']==\'unique_catalogue_companion\',\'Missing/ambiguous companion\')\n            summary,n,path,r=process_granule(join,lookup,session,original_cache,scratch,derived,method_hash,region)\n            summaries.append(summary);native.append(n);clear_paths[join[\'fire_id\']]=path;receipts.extend(r)\n            print(\'Processed\',day,lookup[join[\'fire_id\']][\'name\'],flush=True)\n        except Exception as error:\n            failures.append({\'fire_id\':join[\'fire_id\'],\'error_type\':type(error).__name__,\'reason\':str(error)[:240]})\n            if isinstance(error,e.ScienceAccessRequired):access_denied=True;break\n        write_json(folder/\'failures.json\',failures)\n    archive=pd.DataFrame(inputs[\'standard_observations\']);archive[\'acq_datetime\']=pd.to_datetime(archive.acq_datetime,utc=True);archive=archive[archive.acq_datetime.dt.strftime(\'%Y-%m-%d\').eq(day)].reset_index(drop=True)\n    n=pd.concat(native,ignore_index=True) if native else pd.DataFrame(columns=NATIVE_COLUMNS);typed=attributed_native(n,archive)\n    features,labels=daily_tables(slot,inventory,summaries,typed,archive)\n    # Write source-only features before diagnostic reference-conditioned matching.\n    frames={\'noaa20_features\':features,\'reference_labels\':labels,\'native_pixel_audit\':typed,\'granule_audit\':pd.DataFrame(summaries)}\n    membership=p.group_passes([r for r in inventory[\'records\'] if r[\'date\']==day]);passes=pass_inventory(membership,summaries)\n    frames.update(pass_membership=membership,pass_inventory=passes)\n    try:diag,pairs=geometry_audit(membership,passes,clear_paths)\n    except Exception as error:\n        diag=pd.DataFrame();pairs=pd.DataFrame();failures.append({\'stage\':\'geometry_diagnostic\',\'error_type\':type(error).__name__,\'reason\':str(error)[:240]})\n    frames.update(all_temporal_candidate_diagnostics=diag,geometry_qualified_pairs=pairs,unmatched_standard_observations=archive[~archive.observation_id.isin(typed.archive_observation_id.dropna())])\n    for name,frame in frames.items():frame.to_parquet(folder/(name+\'.parquet\'),index=False)\n    write_json(folder/\'download_receipts.json\',receipts);write_json(folder/\'failures.json\',failures)\n    all_done=len(summaries)==len(joins);files=[name+\'.parquet\' for name in frames]+[\'download_receipts.json\',\'failures.json\']\n    checkpoint(folder,\'checkpoint.json\',signature,files,all_planned_granules_processed=all_done and not failures,access_denied=access_denied,\n      planned_fire_granules=len(joins),processed_fire_granules=len(summaries),date=day)\n    return json.loads((folder/\'checkpoint.json\').read_text())\n\ndef run_batch(inputs,years,session,original_cache,scratch,work,method_hash):\n    scratch=Path(scratch);scratch.mkdir(parents=True,exist_ok=True);Path(work).mkdir(parents=True,exist_ok=True);out=[]\n    for slot in inputs[\'design\'][\'dates\']:\n        if slot[\'year\'] not in years:continue\n        result=process_date(slot,inputs,session,original_cache,scratch,work,method_hash);out.append(result)\n        print(\'Date checkpoint\',slot[\'date\'],\'granules\',result[\'processed_fire_granules\'],\'/\',result[\'planned_fire_granules\'],flush=True)\n        if result[\'access_denied\']:break\n    return {\'years\':list(years),\'dates_attempted\':len(out),\'access_denied\':any(x[\'access_denied\'] for x in out),\'date_audits_complete\':sum(x[\'all_planned_granules_processed\'] for x in out)}\n\ndef fit_candidate(x,y,months,kind):\n    x=np.asarray(x,float);y=np.asarray(y,float)\n    if not len(y) or y.sum()<=0:return None\n    if kind==\'seasonal_baseline\':\n        overall=float(y.mean());return {\'kind\':kind,\'rates\':{m:float((y[np.asarray(months)==m].sum()+3*overall)/((np.asarray(months)==m).sum()+3)) for m in range(1,13)}}\n    if kind in [\'global_ratio\',\'seasonal_ratio\']:\n        if x.sum()<=0:return None\n        ratio=float(y.sum()/x.sum());model={\'kind\':kind,\'ratio\':ratio}\n        if kind==\'seasonal_ratio\':\n            prior=max(1.,float(np.median(x[x>0]))*3);model[\'rates\']={m:float((y[np.asarray(months)==m].sum()+prior*ratio)/(x[np.asarray(months)==m].sum()+prior)) for m in range(1,13)}\n        return model\n    if kind==\'positive_affine\':\n        a,b=nnls(np.column_stack([np.ones(len(x)),x]),y)[0];return {\'kind\':kind,\'intercept\':float(a),\'slope\':float(b)}\n    raise ValueError(\'Unsupported candidate\')\n\ndef predict(model,features):\n    # Whitelist the source-only columns; no reference rows/availability read here.\n    x=features.noaa20_screened_vegetation_active_cells.to_numpy(float);m=features.month_number.to_numpy(int)\n    if model[\'kind\']==\'global_ratio\':return x*model[\'ratio\']\n    if model[\'kind\']==\'seasonal_ratio\':return x*np.array([model[\'rates\'][int(v)] for v in m])\n    if model[\'kind\']==\'seasonal_baseline\':return np.array([model[\'rates\'][int(v)] for v in m])\n    return model[\'intercept\']+model[\'slope\']*x\n\ndef evaluate_outage_proxy(features,labels):\n    p.require(set(features.columns)==set(FEATURE_COLUMNS),\'Unexpected predictor schema; reference-derived fields prohibited\')\n    p.require(set(labels.columns)==set(LABEL_COLUMNS),\'Unexpected reference schema\')\n    p.require(not features.date.duplicated().any() and not labels.date.duplicated().any(),\'Repeated date unit\')\n    for frame,flag,column in [(features,\'noaa20_input_available\',\'noaa20_screened_vegetation_active_cells\'),(labels,\'reference_available\',\'snpp_screened_vegetation_active_cells\')]:\n        values=pd.to_numeric(frame.loc[frame[flag],column],errors=\'raise\').to_numpy(float);p.require(np.isfinite(values).all() and (values>=0).all(),\'Available activity must have finite nonnegative values\')\n    f=features.merge(labels,on=\'date\',how=\'left\',validate=\'one_to_one\');ycol=\'snpp_screened_vegetation_active_cells\';xcol=\'noaa20_screened_vegetation_active_cells\'\n    both=f.noaa20_input_available&f.reference_available.fillna(False);fit=f[both&f.partition.eq(\'fit\')].copy();cal=f[both&f.partition.eq(\'calibration\')].copy()\n    test_features=features[features.partition.eq(\'evaluation\')&features.noaa20_input_available].copy();audit={\'fit_dates\':list(fit.date),\'calibration_dates\':list(cal.date),\'evaluation_source_dates\':list(test_features.date),\'status\':\'insufficient_fit_support\'}\n    p.require(not (set(fit.date)&set(cal.date) or set(fit.date)&set(test_features.date) or set(cal.date)&set(test_features.date)),\'Partition leakage\')\n    if test_features.empty:\n        audit[\'status\']=\'no_source_available_evaluation_dates\';return pd.DataFrame(),pd.DataFrame(),audit\n    if len(fit)<12 or pd.to_datetime(fit.date).dt.year.nunique()<3 or fit[ycol].sum()<=0:return pd.DataFrame(),pd.DataFrame(),audit\n    years=pd.to_datetime(fit.date).dt.year;scores=[]\n    for order,kind in enumerate(POLICY[\'candidate_models\']):\n        error=total=0.;supported=True\n        for year in sorted(years.unique()):\n            a=fit[years.ne(year)];b=fit[years.eq(year)];model=fit_candidate(a[xcol],a[ycol],a.month_number,kind)\n            if model is None:supported=False;break\n            error+=float(np.abs(predict(model,b[FEATURE_COLUMNS])-b[ycol]).sum());total+=float(b[ycol].sum())\n        score=error/total if supported and total>0 else np.inf;scores.append((score,order,kind))\n    best=min(scores)\n    if not np.isfinite(best[0]):return pd.DataFrame(),pd.DataFrame(),audit\n    kind=best[2];model=fit_candidate(fit[xcol],fit[ycol],fit.month_number,kind);baseline=fit_candidate(fit[xcol],fit[ycol],fit.month_number,\'seasonal_baseline\')\n    audit.update(status=\'exploratory_research_evaluated\',selected_model=kind,inner_wape=best[0],candidate_scores={k:float(s) if np.isfinite(s) else None for s,_,k in scores},model=model)\n    test_labels=labels.set_index(\'date\');rows=[]\n    for role,obj in [(\'selected_transfer\',model),(\'native_source_identity\',None),(\'seasonal_baseline\',baseline)]:\n        cp=cal[xcol].to_numpy(float) if obj is None else predict(obj,cal[FEATURE_COLUMNS]);tp=test_features[xcol].to_numpy(float) if obj is None else predict(obj,test_features)\n        residual=np.abs(cal[ycol].to_numpy(float)-cp)/np.sqrt(1+cp);radii={}\n        for level in [80,90]:\n            rank=math.ceil((len(residual)+1)*level/100);radii[level]=float(np.sort(residual)[rank-1]) if len(residual)>=12 and rank<=len(residual) else None\n        for index,row in enumerate(test_features.itertuples()):\n            label=test_labels.loc[row.date] if row.date in test_labels.index else None;available=label is not None and bool(label.reference_available);point=float(tp[index])\n            result={\'date\':row.date,\'role\':role,\'model\':kind if role==\'selected_transfer\' else role,\'prediction\':point,\'reference_available\':available,\'observed\':float(label[ycol]) if available else None,\'prediction_scope\':POLICY[\'target\']}\n            for level,q in radii.items():\n                half=q*np.sqrt(1+point) if q is not None else None;result[\'lower_\'+str(level)]=max(0.,point-half) if half is not None else None;result[\'upper_\'+str(level)]=point+half if half is not None else None\n            result[\'interval_status\']=\'exploratory_date_calibration\' if all(v is not None for v in radii.values()) else \'withheld_insufficient_calibration_dates\';rows.append(result)\n    predictions=pd.DataFrame(rows);metrics=[]\n    for role,group in predictions.groupby(\'role\',sort=False):\n        valid=group[group.reference_available];den=valid.observed.sum();error=(valid.prediction-valid.observed).abs();r={\'role\':role,\'source_available_evaluation_dates\':len(group),\'scored_evaluation_dates\':len(valid),\'wape\':float(error.sum()/den) if den>0 else None,\'mae\':float(error.mean()) if len(valid) else None}\n        for level in [80,90]:\n            a=valid[valid[\'lower_\'+str(level)].notna()];r[\'interval_dates_\'+str(level)]=len(a);r[\'coverage_\'+str(level)]=float(a.observed.between(a[\'lower_\'+str(level)],a[\'upper_\'+str(level)]).mean()) if len(a) else None;r[\'mean_width_\'+str(level)]=float((a[\'upper_\'+str(level)]-a[\'lower_\'+str(level)]).mean()) if len(a) else None\n        metrics.append(r)\n    return predictions,pd.DataFrame(metrics),audit\n\ndef finalize(inputs,work,output,method_hash):\n    work=Path(work);output=Path(output);output.mkdir(parents=True,exist_ok=True);features=[];labels=[];dates=[];failures=[];receipts=[]\n    combined={name:[] for name in [\'native_pixel_audit\',\'granule_audit\',\'pass_membership\',\'pass_inventory\',\'all_temporal_candidate_diagnostics\',\'geometry_qualified_pairs\',\'unmatched_standard_observations\']}\n    for slot in inputs[\'design\'][\'dates\']:\n        day=slot[\'date\'];folder=work/\'dates\'/day;signature={\'method_hash\':method_hash,\'date\':day}\n        if checkpoint_valid(folder,\'checkpoint.json\',signature):\n            cp=json.loads((folder/\'checkpoint.json\').read_text());features.append(pd.read_parquet(folder/\'noaa20_features.parquet\'));labels.append(pd.read_parquet(folder/\'reference_labels.parquet\'));dates.append({**slot,\'status\':\'processed\' if cp[\'all_planned_granules_processed\'] else \'partial\',\'processed_fire_granules\':cp[\'processed_fire_granules\'],\'planned_fire_granules\':cp[\'planned_fire_granules\']})\n            failures.extend([{\'date\':day,**x} for x in json.loads((folder/\'failures.json\').read_text())]);receipts.extend(json.loads((folder/\'download_receipts.json\').read_text()))\n            for name in combined:combined[name].append(pd.read_parquet(folder/(name+\'.parquet\')))\n        else:\n            dates.append({**slot,\'status\':\'not_processed\'});features.append(pd.DataFrame([{\'date\':day,\'partition\':slot[\'partition\'],\'month_number\':int(day[5:7]),\'noaa20_input_available\':False,\'noaa20_screened_vegetation_active_cells\':None,\'noaa20_native_pixels\':None,\'noaa20_nominal_clear_centres\':None,\'noaa20_status\':\'not_processed\'}],columns=FEATURE_COLUMNS));labels.append(pd.DataFrame([{\'date\':day,\'reference_available\':False,\'snpp_screened_vegetation_active_cells\':None,\'snpp_status\':\'not_processed\'}],columns=LABEL_COLUMNS))\n    features=pd.concat(features,ignore_index=True);labels=pd.concat(labels,ignore_index=True);predictions,metrics,audit=evaluate_outage_proxy(features,labels)\n    frames={\'noaa20_features\':features,\'reference_labels\':labels,\'evaluation_predictions\':predictions,\'evaluation_metrics\':metrics,\'date_status\':pd.DataFrame(dates)}\n    frames.update({name:pd.concat(chunks,ignore_index=True) if chunks else pd.DataFrame() for name,chunks in combined.items()})\n    for name,frame in frames.items():frame.to_parquet(output/(name+\'.parquet\'),index=False)\n    write_json(output/\'split_and_model_audit.json\',audit);write_json(output/\'failures.json\',failures);write_json(output/\'download_receipts.json\',receipts);write_json(output/\'method.json\',POLICY)\n    report={\'study_id\':POLICY[\'study_id\'],\'method_hash\':method_hash,\'dates_planned\':len(dates),\'dates_processed\':sum(x[\'status\']==\'processed\' for x in dates),\'date_failures\':len(failures),\n      \'fit_source_dates_available\':int((features.partition.eq(\'fit\')&features.noaa20_input_available).sum()),\'calibration_source_dates_available\':int((features.partition.eq(\'calibration\')&features.noaa20_input_available).sum()),\'evaluation_source_dates_available\':int((features.partition.eq(\'evaluation\')&features.noaa20_input_available).sum()),\n      \'evaluation_dates_withheld\':records(features[features.partition.eq(\'evaluation\')&~features.noaa20_input_available][[\'date\',\'noaa20_status\']]),\n      \'status\':\'exploratory_research_only\',\'model_evaluation_status\':audit[\'status\'],\'metrics\':records(metrics),\'coverage_verified\':False,\'replacement_validated\':False,\'production_export_enabled\':False,\'frontend_changed\':False,\n      \'limitations\':[\'Sampled dates do not validate monthly/annual uncertainty or early history\',\'Geometry matching is centre proximity, not physical footprint coverage\',\'Reference-conditioned support never enters outage predictor features\',\'Archive/type reconciliation and source availability can withhold dates\',\'Actual outage covariate shift remains unverified\',\'Earlier aggregate scores were inspected; not blind external validation\']}\n    write_json(output/\'decision.json\',report);checkpoint(output,\'manifest.json\',{\'method_hash\':method_hash},[path.name for path in output.iterdir() if path.is_file() and path.name!=\'manifest.json\'])\n    return report\n'
base_ns={'__file__':str((STUDY_ROOT/'pipeline'/'base.py').resolve()),'__name__':'study_base'}
exec(compile(BASE_SOURCE,'frozen_base.py','exec'),base_ns)
E=types.SimpleNamespace(**base_ns)
pixel_ns={'__file__':str((STUDY_ROOT/'pipeline'/'pixel.py').resolve()),'__name__':'study_pixel','e':E}
exec(compile(PIXEL_SOURCE.replace('from . import atlas_observation_evidence as e',''),'frozen_pixel.py','exec'),pixel_ns)
P=types.SimpleNamespace(**pixel_ns)
study_ns={'__file__':str((STUDY_ROOT/'pipeline'/'study.py').resolve()),'__name__':'study_validation','e':E,'p':P}
standalone=STUDY_SOURCE.replace('from . import atlas_observation_evidence as e','').replace('from . import atlas_cached_pixel_audit as p','')
exec(compile(standalone,'frozen_multiyear_validation.py','exec'),study_ns)
METHOD_HASH=hashlib.sha256((BASE_SOURCE+PIXEL_SOURCE+STUDY_SOURCE+hashlib.sha256(INPUT_BYTES).hexdigest()).encode()).hexdigest()
WORK=STUDY_ROOT/('run_'+METHOD_HASH[:12]);WORK.mkdir(exist_ok=True)
RESULTS=WORK/'results';RESULTS.mkdir(exist_ok=True)
for name,source in [('base_implementation.py',BASE_SOURCE),('pixel_implementation.py',PIXEL_SOURCE),('study_implementation.py',STUDY_SOURCE)]:
    (RESULTS/name).write_text(source)
print('Resumable run:',WORK.resolve())


Resumable run: /content/drive/MyDrive/CinderTrace_Multiyear_Validation_v1/run_f8aabe8642a8


## 6. Sign in to Earthdata
Enter credentials only at the interactive prompts. Login is not persisted or included in outputs. LP DAAC and LAADS access are both needed. If a batch reports an access denial, rerun this login cell and that same batch cell; successful granules are reused. Later batch cells skip work while `SESSION_READY` is false. Do not share credentials or authenticated redirect URLs.

In [5]:
import earthaccess
auth=earthaccess.login(strategy='interactive',persist=False)
assert auth.authenticated,'Complete Earthdata login before continuing.'
session=earthaccess.get_requests_https_session()
SESSION_READY=True
print('Authenticated for this kernel. No login is written into the research output.')


Authenticated for this kernel. No login is written into the research output.


## 7. Fit batch: 2018–2020
Run this batch and inspect its final status. You can resume the notebook later by repeating setup/login and this cell. Keep the frozen dates and method unchanged. New raw copies are staged temporarily; completed derived checkpoints stay in Drive.

In [6]:
if SESSION_READY:
    batch=study_ns['run_batch'](INPUTS,[2018, 2019, 2020],session,ORIGINAL_CACHE,SCRATCH,WORK,METHOD_HASH)
    print(json.dumps(batch,indent=2))
    if batch['access_denied']:
        SESSION_READY=False
        print('Access denied: rerun step 6, then this batch cell. Verified checkpoints are retained.')
else:
    print('Batch skipped: rerun the Earthdata login cell, then this batch.')


Processed 2018-04-19 VNP14IMG.A2018109.0754.002.2024079212603
Processed 2018-04-19 VNP14IMG.A2018109.2030.002.2024079212605
Date checkpoint 2018-04-19 granules 2 / 2
Processed 2018-05-27 VNP14IMG.A2018147.0742.002.2024080093609
Processed 2018-05-27 VNP14IMG.A2018147.2018.002.2024080093607
Date checkpoint 2018-05-27 granules 2 / 2
Processed 2018-07-03 VNP14IMG.A2018184.0748.002.2024080195512
Processed 2018-07-03 VNP14IMG.A2018184.2024.002.2024080195510
Processed 2018-07-03 VJ114IMG.A2018184.0654.002.2024008020404
Processed 2018-07-03 VJ114IMG.A2018184.0836.002.2024008020412
Processed 2018-07-03 VJ114IMG.A2018184.1930.002.2024008020408
Date checkpoint 2018-07-03 granules 5 / 5
Processed 2018-09-01 VNP14IMG.A2018244.0724.002.2024081125559
Processed 2018-09-01 VNP14IMG.A2018244.2000.002.2024081125556
Processed 2018-09-01 VJ114IMG.A2018244.0812.002.2024009140509
Processed 2018-09-01 VJ114IMG.A2018244.1906.002.2024009140528
Processed 2018-09-01 VJ114IMG.A2018244.2048.002.2024009143911
Date c

## 8. Calibration batch: 2021–2022
Run this batch and inspect its final status. You can resume the notebook later by repeating setup/login and this cell. Keep the frozen dates and method unchanged. New raw copies are staged temporarily; completed derived checkpoints stay in Drive.

In [9]:
if SESSION_READY:
    batch=study_ns['run_batch'](INPUTS,[2021, 2022],session,ORIGINAL_CACHE,SCRATCH,WORK,METHOD_HASH)
    print(json.dumps(batch,indent=2))
    if batch['access_denied']:
        SESSION_READY=False
        print('Access denied: rerun step 6, then this batch cell. Verified checkpoints are retained.')
else:
    print('Batch skipped: rerun the Earthdata login cell, then this batch.')


Date checkpoint 2021-02-12 granules 5 / 5
Date checkpoint 2021-03-29 granules 5 / 5
Date checkpoint 2021-04-01 granules 5 / 5
Date checkpoint 2021-06-23 granules 5 / 5
Date checkpoint 2021-07-05 granules 6 / 6
Date checkpoint 2021-09-11 granules 6 / 6
Date checkpoint 2021-10-05 granules 6 / 6
Date checkpoint 2021-12-15 granules 6 / 6
Date checkpoint 2022-02-13 granules 6 / 6
Date checkpoint 2022-03-13 granules 7 / 7
Date checkpoint 2022-05-15 granules 5 / 5
Date checkpoint 2022-06-24 granules 6 / 6
Date checkpoint 2022-07-10 granules 5 / 5
Date checkpoint 2022-08-30 granules 8 / 8
Date checkpoint 2022-11-02 granules 8 / 8
Date checkpoint 2022-12-07 granules 5 / 5
{
  "years": [
    2021,
    2022
  ],
  "dates_attempted": 16,
  "access_denied": false,
  "date_audits_complete": 16
}


## 9. Evaluation batch: 2023–2024
Run this batch and inspect its final status. You can resume the notebook later by repeating setup/login and this cell. Keep the frozen dates and method unchanged. New raw copies are staged temporarily; completed derived checkpoints stay in Drive.

In [7]:
if SESSION_READY:
    batch=study_ns['run_batch'](INPUTS,[2023, 2024],session,ORIGINAL_CACHE,SCRATCH,WORK,METHOD_HASH)
    print(json.dumps(batch,indent=2))
    if batch['access_denied']:
        SESSION_READY=False
        print('Access denied: rerun step 6, then this batch cell. Verified checkpoints are retained.')
else:
    print('Batch skipped: rerun the Earthdata login cell, then this batch.')


Processed 2023-01-26 VNP14IMG.A2023026.0812.002.2024078143015
Processed 2023-01-26 VNP14IMG.A2023026.1906.002.2024078143042
Processed 2023-01-26 VNP14IMG.A2023026.1912.002.2024078143042
Processed 2023-01-26 VNP14IMG.A2023026.2048.002.2024078143045
Processed 2023-01-26 VJ114IMG.A2023026.0718.002.2024095223745
Processed 2023-01-26 VJ114IMG.A2023026.0724.002.2024095223749
Processed 2023-01-26 VJ114IMG.A2023026.2000.002.2024095223749
Date checkpoint 2023-01-26 granules 7 / 7
Processed 2023-03-15 VNP14IMG.A2023074.0812.002.2024078221316
Processed 2023-03-15 VNP14IMG.A2023074.1912.002.2024078221319
Processed 2023-03-15 VNP14IMG.A2023074.2048.002.2024078221320
Processed 2023-03-15 VJ114IMG.A2023074.0724.002.2024097181222
Processed 2023-03-15 VJ114IMG.A2023074.2000.002.2024097181224
Date checkpoint 2023-03-15 granules 5 / 5
Processed 2023-04-30 VNP14IMG.A2023120.0712.002.2024079055448
Processed 2023-04-30 VNP14IMG.A2023120.0848.002.2024079055447
Processed 2023-04-30 VNP14IMG.A2023120.0854.002.

## 10. Build the final evidence tables and run the frozen evaluation
This can also package an honest partial run. Every planned date gets an explicit availability/status row. The two unresolved NOAA-20 dates remain unavailable and do not become zero activity. Models are withheld if fit support is insufficient; intervals are withheld if calibration support is insufficient. Only reference-available evaluation dates are scored. The source predictor's output is not selected using reference availability.

Actual-outage distribution shifts, local observing coverage, early-history transfer and district allocation remain unvalidated. No actual gaps are filled, regardless of these exploratory scores.

In [10]:
DECISION=study_ns['finalize'](INPUTS,WORK,RESULTS,METHOD_HASH)
print(json.dumps(DECISION,indent=2))
from IPython.display import display
display(pd.read_parquet(RESULTS/'date_status.parquet'))
display(pd.read_parquet(RESULTS/'noaa20_features.parquet'))
display(pd.read_parquet(RESULTS/'evaluation_metrics.parquet'))
display(pd.DataFrame(json.loads((RESULTS/'failures.json').read_text())))


{
  "study_id": "lk_viirs_expanded_processing_v1",
  "method_hash": "f8aabe8642a8f6ba67c0221aa73f0dc7b8f778090a5345012fb20cd0716f4349",
  "dates_planned": 54,
  "dates_processed": 54,
  "date_failures": 0,
  "fit_source_dates_available": 20,
  "calibration_source_dates_available": 15,
  "evaluation_source_dates_available": 16,
  "evaluation_dates_withheld": [],
  "status": "exploratory_research_only",
  "model_evaluation_status": "exploratory_research_evaluated",
  "metrics": [
    {
      "role": "selected_transfer",
      "source_available_evaluation_dates": 16,
      "scored_evaluation_dates": 15,
      "wape": 0.7728013606,
      "mae": 5.3065693431,
      "interval_dates_80": 15,
      "coverage_80": 1.0,
      "mean_width_80": 24.106853294,
      "interval_dates_90": 15,
      "coverage_90": 1.0,
      "mean_width_90": 31.45103098
    },
    {
      "role": "native_source_identity",
      "source_available_evaluation_dates": 16,
      "scored_evaluation_dates": 15,
      "wape": 

,date,half_quarter,partition,quarter,selection_basis,year,status,processed_fire_granules,planned_fire_granules
0,2018-04-19,1,fit,2,calendar and availability only; no detection c...,2018,processed,2,2
1,2018-05-27,2,fit,2,calendar and availability only; no detection c...,2018,processed,2,2
2,2018-07-03,1,fit,3,calendar and availability only; no detection c...,2018,processed,5,5
3,2018-09-01,2,fit,3,calendar and availability only; no detection c...,2018,processed,5,5
4,2018-10-22,1,fit,4,calendar and availability only; no detection c...,2018,processed,8,8
5,2018-12-10,2,fit,4,calendar and availability only; no detection c...,2018,processed,5,5
6,2019-02-14,1,fit,1,calendar and availability only; no detection c...,2019,processed,7,7
7,2019-03-08,2,fit,1,calendar and availability only; no detection c...,2019,processed,6,6
8,2019-04-22,1,fit,2,calendar and availability only; no detection c...,2019,processed,7,7
9,2019-05-18,2,fit,2,calendar and availability only; no detection c...,2019,processed,7,7


,date,partition,month_number,noaa20_input_available,noaa20_screened_vegetation_active_cells,noaa20_native_pixels,noaa20_nominal_clear_centres,noaa20_status
0,2018-04-19,fit,4,False,NaN,NaN,NaN,catalogue_absent_not_zero_activity
1,2018-05-27,fit,5,False,NaN,NaN,NaN,catalogue_absent_not_zero_activity
2,2018-07-03,fit,7,True,3.0,3.0,324586.0,recorded_screened_activity_coverage_unverified
3,2018-09-01,fit,9,True,39.0,45.0,453355.0,recorded_screened_activity_coverage_unverified
4,2018-10-22,fit,10,True,0.0,0.0,393590.0,recorded_screened_activity_coverage_unverified
5,2018-12-10,fit,12,True,0.0,0.0,102772.0,recorded_screened_activity_coverage_unverified
6,2019-02-14,fit,2,True,7.0,7.0,542526.0,recorded_screened_activity_coverage_unverified
7,2019-03-08,fit,3,True,20.0,21.0,343900.0,recorded_screened_activity_coverage_unverified
8,2019-04-22,fit,4,True,5.0,8.0,420941.0,recorded_screened_activity_coverage_unverified
9,2019-05-18,fit,5,True,1.0,1.0,608122.0,recorded_screened_activity_coverage_unverified


,role,source_available_evaluation_dates,scored_evaluation_dates,wape,mae,interval_dates_80,coverage_80,mean_width_80,interval_dates_90,coverage_90,mean_width_90
0,selected_transfer,16,15,0.772801,5.306569,15,1.000000,24.106853,15,1.000000,31.451031
1,native_source_identity,16,15,0.815534,5.600000,15,1.000000,24.228792,15,1.000000,32.249745
2,seasonal_baseline,16,15,1.290291,8.860000,15,0.333333,10.768536,15,0.866667,21.290555


""


## 11. Download one results ZIP
Run this after finalization, even for a partial run. Save this notebook with its outputs and add the ZIP to the project folder. It includes the frozen inputs and original catalogue evidence, native pixel audit, pass diagnostics, source-only features, separate reference labels, held-out predictions, calibration bounds, split/model audit, failure states, receipts, methods, versions and hashes. Raw and derived clear-centre caches are excluded. Keep the Drive study workspace for resuming or reviewing centres. Your original frontend remains unchanged.

In [11]:
from datetime import datetime,timezone
import importlib.metadata
assert study_ns['checkpoint_valid'](RESULTS,'manifest.json',{'method_hash':METHOD_HASH}),'Finalize results before downloading.'
versions={name:importlib.metadata.version(name) for name in ['numpy','pandas','pyarrow','shapely','h3','scipy','netCDF4','earthaccess']}
members={'verified_inputs.json':INPUT_BYTES,'runtime_versions.json':(json.dumps(versions,indent=2)+'\n').encode()}
for path in sorted(RESULTS.iterdir()):
    if path.is_file():members['results/'+path.name]=path.read_bytes()
manifest={name:{'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest()} for name,raw in sorted(members.items())}
archive_path=WORK/('cindertrace_atlas_lk_multiyear_validation_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for name,raw in sorted(members.items()):archive.writestr(name,raw)
    archive.writestr('bundle_manifest.json',json.dumps(manifest,indent=2)+'\n')
print('Saved:',archive_path.resolve())
print('SHA256:',hashlib.sha256(archive_path.read_bytes()).hexdigest())
if 'session' in globals():session.close()
if IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))


Saved: /content/drive/MyDrive/CinderTrace_Multiyear_Validation_v1/run_f8aabe8642a8/cindertrace_atlas_lk_multiyear_validation_20261115T075301Z.zip
SHA256: 6598a59cd4282823518b34ae5df20569b4be2b001170c9c7b6b26c54a326c296


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>